# Motion from Scratch

_How to go from pixels to motion._ This notebook runs JavaScript directly, with the **Deno** kernel.

## Set up once

1. Install [Deno](https://docs.deno.com/runtime/getting_started/installation/), then register its Jupyter kernel: `deno jupyter --install`
2. Get the course: `git clone https://github.com/nalediym/motion-from-scratch`
3. Open `notebooks/jupyter-deno.ipynb` in VS Code (with the Jupyter extension) or JupyterLab, and pick the **Deno** kernel.

## How to use it

Run the setup cell. Then for each layer: read the chapter, write your code in the code cell, run it, and run the check cell under it. Your code lives in this notebook; save it as you go.

In [ ]:
// Setup: loads the course helpers. Run once per session.
const base = Deno.cwd().endsWith("notebooks") ? ".." : ".";
(0, eval)(await Deno.readTextFile(`${base}/lib/course.js`));
for (const f of [...MS.COURSE.libs, "lib/png.js", "lib/notebook.js"]) {
  (0, eval)(await Deno.readTextFile(`${base}/${f}`));
}
for (const l of MS.COURSE.layers.filter(l => l.ready)) (0, eval)(await Deno.readTextFile(`${base}/${MS.COURSE.files(l).mystery}`));
async function check(layerId) {
  const layer = MS.COURSE.layers.find(l => l.id === layerId);
  MS.loadTests(layerId, await Deno.readTextFile(`${base}/${MS.COURSE.files(layer).test}`));
  const r = MS.runLayerReport(layerId);
  return Deno.jupyter.display({ "text/html": r.html, "text/plain": `${r.passed} / ${r.total} passing` }, { raw: true });
}
async function checkpoint(layerId) {
  const layer = MS.COURSE.layers.find(l => l.id === layerId);
  MS.loadTests(layerId, await Deno.readTextFile(`${base}/${MS.COURSE.files(layer).test}`));
  const cp = MS.runCheckpoint(layerId, MS.runTests(layerId));
  return Deno.jupyter.display({ "text/html": MS.checkpointHTML(cp), "text/plain": MS.checkpointText(cp) }, { raw: true });
}
"Course helpers loaded"

## Layer 00 · Pixels

_A framebuffer is memory you can see._

**You'll write:** `createFrame`, `pixelIndex`, `setPixel`, `getPixel`, `clearFrame` in `src/00-pixels.js`

Every frame of the KineticIntro you'll build in layer 09 is 1080 × 1920 pixels. Before anything can move, you need somewhere to put those pixels and a way to change them one at a time. That place is called a **framebuffer**, and by the end of this layer you'll have written one.

### Under the hood

#### A pixel is three or four numbers

Your screen is a grid of tiny lights. Each one mixes red, green and blue light, and how bright each color glows is a number from 0 (off) to 255 (full). Why 255? Each channel gets one **[byte](https://en.wikipedia.org/wiki/Byte)**: 8 bits, which can hold 2⁸ = 256 different values, 0 through 255.

```text
  [255,   0,   0]  red          [  0,   0,   0]  black (all off)
  [255, 255, 255]  white        [ 91, 140, 255]  Motion Studio's blue
```

A fourth number, **[alpha](https://en.wikipedia.org/wiki/Alpha_compositing)**, says how opaque the pixel is: 255 is solid, 0 is invisible. Alpha doesn't matter yet, but layer 01 is all about it. So one pixel is 4 bytes: **R, G, B, A**.

#### Memory is one long line

A picture is 2-D, but computer memory is a single long row of bytes. So you have to decide how to lay the grid out in a line. Nearly everything (browsers, image files, GPUs) uses the same answer: **row by row, left to right**, 4 bytes per pixel.

Here's a frame 4 pixels wide and 3 tall, with each cell showing where that pixel starts in the line:

```text
            x = 0   x = 1   x = 2   x = 3
          ┌───────┬───────┬───────┬───────┐
  y = 0   │   0   │   4   │   8   │  12   │
          ├───────┼───────┼───────┼───────┤
  y = 1   │  16   │  20   │  24   │  28   │
          ├───────┼───────┼───────┼───────┤
  y = 2   │  32   │  36   │  40   │  44   │
          └───────┴───────┴───────┴───────┘

  data: [ R G B A  R G B A  R G B A  R G B A | R G B A ... ]
          └ (0,0) ┘└ (1,0) ┘└ (2,0) ┘└ (3,0) ┘ └ (0,1) ...
          index 0   index 4   index 8   index 12  index 16
```

Look at the numbers for a while before writing `pixelIndex`. Going one step right adds the same amount each time. Going one step down adds a different, bigger amount. What is each amount, and why? If you can say that out loud, the formula writes itself.

> **Study buddy:** "Don't give me code. Ask me questions until I can explain what pixelIndex multiplies y by, and why there's a × 4." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Don%27t%20give%20me%20code.%20Ask%20me%20questions%20until%20I%20can%20explain%20what%20pixelIndex%20multiplies%20y%20by%2C%20and%20why%20there%27s%20a%20%C3%97%204.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Don%27t%20give%20me%20code.%20Ask%20me%20questions%20until%20I%20can%20explain%20what%20pixelIndex%20multiplies%20y%20by%2C%20and%20why%20there%27s%20a%20%C3%97%204.)

#### Why a Uint8ClampedArray

JavaScript has **[typed arrays](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Typed_arrays)**: fixed-size blocks of raw numbers, much faster than normal arrays because every slot is the same type. [`Uint8ClampedArray`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Uint8ClampedArray) means:

- **Uint8**: each slot is an unsigned 8-bit integer, 0–255. Exactly one channel.
- **Clamped**: if you store something out of range, it's pinned to the nearest edge instead of wrapping around. Store 300 and you get 255; store −5 and you get 0; store 127.6 and it rounds to 128. The exact rule is the [ToUint8Clamp](https://tc39.es/ecma262/#sec-touint8clamp) operation in the ECMAScript specification, the document that defines JavaScript.

The browser uses this exact type for the pixels of a canvas ([`ImageData.data`](https://developer.mozilla.org/en-US/docs/Web/API/ImageData/data)), which is why you're using it. The HTML standard [defines `ImageData`](https://html.spec.whatwg.org/multipage/imagebitmap-and-animations.html#the-imagedata-interface) as RGBA bytes stored left to right, row by row from the top: the same layout you're about to build. Compare it with its unclamped cousin [`Uint8Array`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Uint8Array), where 300 wraps around to 44 (300 − 256).

> **Study buddy:** "Quiz me on Uint8ClampedArray versus Uint8Array with three predict-the-output questions." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Quiz%20me%20on%20Uint8ClampedArray%20versus%20Uint8Array%20with%20three%20predict-the-output%20questions.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20Quiz%20me%20on%20Uint8ClampedArray%20versus%20Uint8Array%20with%20three%20predict-the-output%20questions.)

#### The edge of the world

What should happen if someone calls `setPixel(frame, 4, 0, ...)` on a frame that's 4 pixels wide? There is no column 4. But the index formula will still happily compute a number, and that number lands somewhere real:

```text
  setPixel(frame, x = 4, y = 0)  on a 4-wide frame

          x = 0   x = 1   x = 2   x = 3   (x = 4?)
  y = 0   ░░░░░░  ░░░░░░  ░░░░░░  ░░░░░░  ← you meant here
  y = 1   ██████  ░░░░░░  ░░░░░░  ░░░░░░
          └─ but the formula points here: (0, 1)
```

This wrap-around bug is real: shapes that hang off the right edge reappear on the left, one row down. `setPixel` must check bounds itself, and later layers slide things off-screen all the time.

#### Getting it on screen

Your frame is just numbers in memory. To see it, `lib/present.js` (provided) wraps them in an [`ImageData`](https://developer.mozilla.org/en-US/docs/Web/API/ImageData) and copies them into a [`<canvas>`](https://developer.mozilla.org/en-US/docs/Web/API/Canvas_API) with [`putImageData`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/putImageData):

```text
  your frame.data ──► new ImageData(data, w, h) ──► ctx.putImageData(...)
        (CPU memory)        (browser wrapper)          │
                                                       ▼
                     display ◄── GPU ◄── compositor ◄── canvas
               (refreshes ~60 times a second)
```

You don't need to write that part. The important idea is that the screen only ever shows what's in memory, and drawing just means changing numbers.

### Your turn

Work through `src/00-pixels.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **createFrame.** How many bytes does a `width × height` frame need? What does a brand-new typed array contain? What counts as a "positive whole number", and is `2.5` one? ([`Number.isInteger`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Number/isInteger) may help.)
2. **pixelIndex.** Use the grid above. What do you multiply `y` by to skip whole rows? Why the final `× 4`?
3. **setPixel.** What order do you do things in: round ([`Math.floor`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Math/floor)), check bounds, compute index, write? What happens if you compute the index before checking bounds? What should alpha be when the color only has three numbers?
4. **getPixel.** Why return a *new* array instead of a piece of `frame.data`? (Hint: what happens if the caller changes it?)
5. **clearFrame.** You could call `setPixel` for every pixel. That works. Is there a way that touches `frame.data` directly with a single loop? Which is faster, and why?

Run the tests after each function. A failing test tells you exactly which pixel was wrong.

Once it works, the test pictures show a checkerboard and a blue-and-orange XOR pattern.

> **Study buddy:** "I'll paste my setPixel. Don't fix it. Tell me which test case would break it, and let me find the bug." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27ll%20paste%20my%20setPixel.%20Don%27t%20fix%20it.%20Tell%20me%20which%20test%20case%20would%20break%20it%2C%20and%20let%20me%20find%20the%20bug.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27ll%20paste%20my%20setPixel.%20Don%27t%20fix%20it.%20Tell%20me%20which%20test%20case%20would%20break%20it%2C%20and%20let%20me%20find%20the%20bug.)



### ✏️ Your code for layer 00

This cell is your `src/00-pixels.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2000%20%28Pixels%29%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 00 · Pixels — a framebuffer: memory you can see.
// Read the chapter first (docs/00-pixels.md). Each comment below is a spec; the tests check it.
//
// A frame is a plain object:
//   { width, height, data }
// where `data` is a Uint8ClampedArray holding 4 numbers per pixel, row by row:
//
//   data:  [ R G B A | R G B A | R G B A | ... ]
//            pixel 0   pixel 1   pixel 2
//
// A color is an array [r, g, b] or [r, g, b, a], each 0–255.
// If alpha is missing, treat it as 255 (fully opaque).
(function (MS) {
  /**
   * Make a new frame, every pixel transparent black [0, 0, 0, 0].
   * Throw an Error if width or height is not a positive whole number.
   * @returns {{ width: number, height: number, data: Uint8ClampedArray }}
   */
  function createFrame(width, height) {
    throw MS.todo("createFrame");
  }

  /**
   * Where does pixel (x, y) start inside frame.data?
   * Return the index of its R value. (G, B and A follow right after it.)
   * Assume x and y are whole numbers inside the frame.
   */
  function pixelIndex(frame, x, y) {
    throw MS.todo("pixelIndex");
  }

  /**
   * Write `color` into pixel (x, y), replacing what was there.
   * - Round fractional coordinates down (2.7 → 2).
   * - If (x, y) is outside the frame, do nothing. Don't throw, and don't
   *   let x = width "wrap around" onto the next row.
   */
  function setPixel(frame, x, y, color) {
    throw MS.todo("setPixel");
  }

  /**
   * Read pixel (x, y) as a new array [r, g, b, a].
   * Round coordinates down like setPixel. Return null if outside the frame.
   */
  function getPixel(frame, x, y) {
    throw MS.todo("getPixel");
  }

  /**
   * Fill every pixel of the frame with `color`.
   */
  function clearFrame(frame, color) {
    throw MS.todo("clearFrame");
  }

  Object.assign(MS, { createFrame, pixelIndex, setPixel, getPixel, clearFrame });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("00")

### Predict, then run

Guess before you run: the point is to catch the places where your mental model and the machine disagree.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: how many bytes does one full-size video frame need?
const myGuess = null; // TODO: your guess, a single number
const frame = MS.createFrame(1080, 1920);
MS.reveal("data.length of a 1080 × 1920 frame", myGuess, frame.data.length);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: what comes back after writing out-of-range numbers?
const myGuess = null; // TODO: your guess as [r, g, b, a]
const f = MS.createFrame(1, 1);
MS.setPixel(f, 0, 0, [300, -5, 127.6]);
MS.reveal("getPixel after setPixel([300, -5, 127.6])", myGuess, MS.getPixel(f, 0, 0));
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: what happens when you write to slot 1.5? (None of your code is involved.)
const myGuess = null; // TODO: your guess as an array of 4 numbers
const a = new Uint8ClampedArray(4);
a[1.5] = 200;
MS.reveal("the array after a[1.5] = 200", myGuess, a);
}


<details><summary>Answers</summary>

1. 8,294,400 (1080 × 1920 × 4), about 8.3 MB. Six seconds at 30 fps is about 1.5 GB, yet the MP4 you export in layer 10 will be a few megabytes. Hold on to that gap; it's what video compression is for.
2. `[255, 0, 128, 255]`. Clamped, rounded, and alpha filled in.
3. `[0, 0, 0, 0]`. A typed array only has whole-number slots, and a write to a fractional index like `1.5` is silently thrown away: no error, no change. This is why `setPixel` must round coordinates. Forget it, and drawing at x = 2.5 quietly does nothing.

Bonus: try `a[0] = 127.5; a[1] = 128.5`. Both become 128. Clamped arrays round exact halves to the nearest even number ("banker's rounding"), so rounding errors don't all lean the same way.

</details>

### Break it

Once your tests pass, break things on purpose. Each **TODO** says what to change in your own code; run the experiment before and after, and explain the difference. Then undo the change and check your tests pass again.

#### 1. Swap x and y



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: where do two pixels start, before and after the swap?
// TODO(break-it), in src/00-pixels.js → pixelIndex:
//   swap x and y in your formula. Change nothing else.
const f = MS.createFrame(4, 2);   // 4 wide, 2 tall: data has 32 slots, 0 to 31
console.log("pixelIndex(3, 0) =", MS.pixelIndex(f, 3, 0));
console.log("pixelIndex(0, 1) =", MS.pixelIndex(f, 0, 1));
// TODO: note both numbers, make the change, run again.
// Is either new number outside the 32 slots? What would setPixel do with it?
}


Then run the tests and look at the XOR pattern. Which tests catch the swap, and what happened to the picture?

#### 2. Delete the bounds check



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: draw one pixel just past the right edge.
// TODO(break-it), in src/00-pixels.js → setPixel:
//   comment out your check that (x, y) is inside the frame.
const f = MS.createFrame(4, 3);
MS.setPixel(f, 4, 0, [255, 255, 255]);   // x = width: there is no column 4
console.log(MS.asciiFrame(f));
// TODO: before the change you should see only dots (·).
// After it, which pixel lit up, and why that one?
}


#### 3. Lose the clamping



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: out-of-range values, with and without clamping.
// TODO(break-it), in src/00-pixels.js → createFrame:
//   use Uint8Array instead of Uint8ClampedArray.
const f = MS.createFrame(1, 1);
MS.setPixel(f, 0, 0, [300, -5, 127.6]);
console.log(MS.getPixel(f, 0, 0));
// TODO: after the change, explain each of the three numbers.
// Hint: 256 possible values wrap around like a clock face.
}


Which test fails with `Uint8Array`, and why would wrapping be a disaster for colors?

### Checkpoint

Your layer is **complete** when three things are true:

1. **All 11 tests pass.** That unlocks the checkpoint.
2. **The mystery picture comes out right.** The checkpoint runs your functions on a scrambled list of drawing commands full of this chapter's traps: fractional coordinates, pixels one step past the right edge, colors out of range. Get every rule right and a hidden word appears. Get one wrong and the picture shows it, with a hint about which rule to look at (never the fix).
3. **Three questions are answered correctly** below. You may type an expression instead of a number, like `10 * 10`, or compute an answer with your own functions.



In [ ]:
// Checkpoint 00 · Pixels. Replace each null with your answer.
MS.answers("00", {
  // 1. In a frame 12 pixels wide, at which index of data does pixel (7, 3) start?
  q1: null,
  // 2. How many bytes do 90 frames of 1280 × 720 take, before any compression?
  q2: null,
  // 3. You store -12, 256, 2.5 and 3.5 into a new Uint8ClampedArray(4).
  //    What does it hold? Answer as an array of 4 numbers.
  q3: null,
});

In [ ]:
await checkpoint("00")


### In the real world

- Video cards keep the image being shown in a region of memory literally called the framebuffer, scanned out to the display many times a second.
- Remotion doesn't touch pixels directly. It draws with the browser, then takes a screenshot of each frame, which is a full framebuffer like yours. You'll see that in layer 10.

### Going further (optional)

- **Read the original.** [ECMAScript: TypedArray objects](https://tc39.es/ecma262/#sec-typedarray-objects): how typed arrays work, including why a write to index `1.5` is ignored.
- Write `fillFrameFast(frame, color)` using [`Uint32Array`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Uint32Array) over the same [buffer](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/ArrayBuffer) (`new Uint32Array(frame.data.buffer)`) to set a whole pixel in one write. Time it against `clearFrame` on a 1080 × 1920 frame. Then look up **[endianness](https://en.wikipedia.org/wiki/Endianness)** to understand why the byte order inside that 32-bit number looks backwards.


## Layer 01 · Shapes & blending

_Deciding which pixels a shape covers, and what happens where shapes overlap._

**You'll write:** `hex`, `blendPixel`, `fillRect`, `fillCircle`, `mixColor`, `fillVerticalGradient`, `fillRadialGradient` in `src/01-shapes.js`
**Builds on:** layer 00

KineticIntro has a glowing radial background, words that fade in and out, and a light sweep that brightens the logo as it passes. All three are this layer: filling shapes, mixing colors, and **blending** a see-through color over what's already there.

### Under the hood

#### Colors as text: hex

Designers write colors like `#5B8CFF` (the CSS [hex color](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/hex-color) notation). That's three bytes written in base 16 ([hexadecimal](https://en.wikipedia.org/wiki/Hexadecimal)), two digits each:

```text
   #  5B   8C   FF
      │    │    └─ blue  = 0xFF = 15×16 + 15 = 255
      │    └────── green = 0x8C =  8×16 + 12 = 140
      └─────────── red   = 0x5B =  5×16 + 11 =  91
```

JavaScript can read base-16 text for you (look up [`parseInt`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/parseInt)'s second argument, the radix). There are two common approaches: cut the string into three 2-character pieces, or read all six digits as one number and pull bytes out of it with bit shifts ([`>>`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/Right_shift)) and masks ([`& 255`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/Bitwise_AND)). Either is fine. The second shows you how colors are often packed in memory.

#### Rasterizing: which pixels does a shape cover?

A rectangle from x = 2 to x = 6 is an idea with perfect edges. The screen only has whole pixels. Turning shapes into pixels is called **[rasterization](https://en.wikipedia.org/wiki/Rasterisation)**, and the rule almost every renderer uses is: _a pixel belongs to the shape if its **center** is inside._

```text
        0     1     2     3     4     5     6     7
     ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
     │  ·  │  ·  │  ■  │  ■  │  ■  │  ■  │  ·  │  ·  │   rect x = 2 .. 6
     └─────┴─────┴─────┴─────┴─────┴─────┴─────┴─────┘
                 ↑ centers at 2.5, 3.5, 4.5, 5.5 are inside
                                         ↑ 6.5 is not
```

For rectangles, testing centers is the same as rounding the edges, which is what the spec asks for.

The right edge is **exclusive** (6 is out). This "half-open" rule means two rectangles that share an edge (one from 2 to 6, the next from 6 to 9) meet perfectly, with no gap and no pixel painted twice. You'll see why that matters in **Break it**.

> **Study buddy:** "Ask me to predict which pixels fillRect(1.5, 0, 2, 1) covers, then tell me whether my reasoning (not just my answer) was right." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20to%20predict%20which%20pixels%20fillRect%281.5%2C%200%2C%202%2C%201%29%20covers%2C%20then%20tell%20me%20whether%20my%20reasoning%20%28not%20just%20my%20answer%29%20was%20right.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20to%20predict%20which%20pixels%20fillRect%281.5%2C%200%2C%202%2C%201%29%20covers%2C%20then%20tell%20me%20whether%20my%20reasoning%20%28not%20just%20my%20answer%29%20was%20right.)

#### Circles: distance from the center

A pixel is inside a circle if its center is no farther than `r` from the circle's center. That's Pythagoras: `dx² + dy² ≤ r²`. (You can skip the square root by comparing squared distances. Why does that give the same answer?)

```text
          · · · · · · · · ·
          · · · ■ ■ ■ · · ·         every ■ passes  dx² + dy² ≤ r²
          · · ■ ■ ■ ■ ■ · ·
          · · ■ ■ ● ■ ■ · ·         ● = center
          · · ■ ■ ■ ■ ■ · ·
          · · · ■ ■ ■ · · ·         ┌─────────┐  only test pixels
          · · · · · · · · ·         └─────────┘  inside this box
```

You *could* test every pixel in the frame. For a small circle on a 1080 × 1920 frame that's 2 million tests to draw maybe 300 pixels. Instead, only visit the circle's **bounding box**: from `cx − r` to `cx + r` in each direction.

Jagged edges like these are called **aliasing**. Real renderers soften them with **[antialiasing](https://en.wikipedia.org/wiki/Spatial_anti-aliasing)**: a pixel that's 40% covered gets 40% of the color. See **Going further**.

#### Alpha blending: painting with see-through paint

Here's the big one. When you paint a half-transparent red over a blue pixel, the result is a mix. When you paint it over an empty, fully transparent pixel, it should still be red, just see-through.

This was worked out in 1984 by Thomas Porter and Tom Duff at Lucasfilm, in a paper called ["Compositing Digital Images"](https://keithp.com/~keithp/porterduff/p253-porter.pdf) (SIGGRAPH '84, PDF). It's how film effects layered spaceships over backgrounds, and how every browser layers one element over another. Their "over" operation, with alphas written as 0 to 1:

```text
  a = alpha of the new color (source)     b = alpha of the existing pixel (destination)

  outAlpha = a + b × (1 − a)
  outRGB   = ( newRGB × a  +  oldRGB × b × (1 − a) ) / outAlpha
```

Read it as coverage. The new paint covers fraction `a` of the pixel. Of what's left, `(1 − a)`, the old pixel covers fraction `b`. The color is the average of the two, weighted by how much each one covers. The **divide by outAlpha** turns "amount of light" back into "color".

```text
  50% red over opaque blue:              50% red over nothing:
  outAlpha = 0.5 + 1 × 0.5 = 1           outAlpha = 0.5 + 0 = 0.5
  red  = (255×0.5 + 0) / 1   ≈ 128       red  = (255×0.5) / 0.5 = 255   ← still pure red
  blue = (0 + 255×1×0.5) / 1 ≈ 127       alpha = 0.5 → 128
```

Colors stored this way are called **straight alpha**. Many renderers store colors already multiplied by alpha instead (**premultiplied alpha**), because then the formula needs no division. Porter and Duff wrote "over" for premultiplied colors in the paper; the formula above is the same equation with the division put back so it works on straight colors. Browsers follow the [W3C Compositing and Blending spec](https://www.w3.org/TR/compositing-1/#porterduffcompositingoperators_srcover), which names this operator **source-over**.

> **Study buddy:** "Without giving me code, walk me through Porter–Duff 'over' with three cases I can check by hand, then explain straight versus premultiplied alpha with an analogy." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20Without%20giving%20me%20code%2C%20walk%20me%20through%20Porter%E2%80%93Duff%20%27over%27%20with%20three%20cases%20I%20can%20check%20by%20hand%2C%20then%20explain%20straight%20versus%20premultiplied%20alpha%20with%20an%20analogy.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Here%27s%20what%20I%20want%20help%20with%3A%20Without%20giving%20me%20code%2C%20walk%20me%20through%20Porter%E2%80%93Duff%20%27over%27%20with%20three%20cases%20I%20can%20check%20by%20hand%2C%20then%20explain%20straight%20versus%20premultiplied%20alpha%20with%20an%20analogy.)

#### Gradients: one color per position

A gradient answers one question per pixel: _how far along am I, from 0 to 1?_ Then it mixes the two colors by that amount:

```text
  mixColor(a, b, t) = a + (b − a) × t      (for each of R, G, B and A)

  t = 0          t = 0.25        t = 0.5         t = 0.75        t = 1
  ██ a           ██              ██ halfway      ██              ██ b
```

A vertical gradient gets `t` from the row. A radial gradient gets `t` from distance to a center point. Dividing each axis by its own radius (`rx`, `ry`) stretches the circle into an ellipse.

That's exactly what CSS does: [`radial-gradient()`](https://developer.mozilla.org/en-US/docs/Web/CSS/gradient/radial-gradient), defined in the [CSS Images spec](https://www.w3.org/TR/css-images-3/#radial-gradients), which says the first size is the horizontal radius and the second the vertical, each as a percentage of the box. KineticIntro's background is written in CSS as:

```text
  radial-gradient(70% 60% at 50% 38%, #1A2740, #0B0D12)
                  └rx  ry┘   └cx  cy┘   inner     outer
  on a 1080 × 1920 frame:  rx = 756   ry = 1152   cx = 540   cy = 729.6
```

### Your turn

1. **hex.** What should happen with `"#FFF"`, `"blue"`, or `"#12345G"`? A [regular expression](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Regular_expressions) like `/^#?[0-9a-f]{6}$/i` is one way to check. Can you explain each part of it?
2. **blendPixel.** Turn both alphas into 0–1 first. Handle the case where `outAlpha` is 0. What would dividing by 0 put in the pixel?
3. **fillRect.** Round the four edges, then clip them to the frame *before* looping. Why is clipping first better than letting `blendPixel` ignore off-screen pixels?
4. **fillCircle.** What does your bounding box look like when the circle hangs off the frame?
5. **mixColor, fillVerticalGradient, fillRadialGradient.** Where is the middle of row `y`? Why sample there instead of at its top edge?

Once it works, the test pictures show overlapping blended shapes and KineticIntro's exact background: the first real piece of the recreation.



### ✏️ Your code for layer 01

This cell is your `src/01-shapes.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2001%20%28Shapes%20%26%20blending%29%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 01 · Shapes & blending — rectangles, circles, alpha, gradients.
// Read the chapter first (docs/01-shapes.md). Each comment below is a spec; the tests check it.
// You can (and should) use your layer 00 functions here.
(function (MS) {
  /**
   * Turn a CSS-style hex string into a color array.
   *   hex("#5B8CFF")      → [91, 140, 255, 255]
   *   hex("5b8cff", 128)  → [91, 140, 255, 128]
   * Accept upper or lower case, with or without "#".
   * Throw an Error for anything that isn't exactly 6 hex digits.
   */
  function hex(str, alpha = 255) {
    throw MS.todo("hex");
  }

  /**
   * Paint `color` OVER pixel (x, y), respecting its alpha ("source-over").
   * With a = alpha of the new color, b = alpha of the existing pixel,
   * both as 0–1:
   *
   *   outAlpha = a + b × (1 − a)
   *   outRGB   = (newRGB × a + oldRGB × b × (1 − a)) / outAlpha
   *
   * Write outAlpha back as 0–255. If outAlpha is 0, the pixel becomes
   * [0, 0, 0, 0]. Round coordinates down; ignore pixels outside the frame.
   */
  function blendPixel(frame, x, y, color) {
    throw MS.todo("blendPixel");
  }

  /**
   * Blend a w × h rectangle whose top-left corner is (x, y).
   * Round the edges to whole pixels first:
   *   left = round(x), right = round(x + w)   (right is exclusive)
   *   top  = round(y), bottom = round(y + h)  (bottom is exclusive)
   * Clip to the frame, so a rectangle hanging off the edge is fine.
   */
  function fillRect(frame, x, y, w, h, color) {
    throw MS.todo("fillRect");
  }

  /**
   * Blend a circle with center (cx, cy) and radius r.
   * A pixel is inside when its CENTER, (px + 0.5, py + 0.5), is at
   * distance <= r from (cx, cy). Only visit pixels in the circle's
   * bounding box, not the whole frame.
   */
  function fillCircle(frame, cx, cy, r, color) {
    throw MS.todo("fillCircle");
  }

  /**
   * Mix two colors: t = 0 gives a, t = 1 gives b, 0.5 is halfway.
   * Mix all four channels (missing alpha = 255). Don't round.
   */
  function mixColor(a, b, t) {
    throw MS.todo("mixColor");
  }

  /**
   * Fill the whole frame from `top` color to `bottom` color.
   * Row y uses t = (y + 0.5) / height, so you sample each row at its middle.
   * Overwrite pixels (setPixel), don't blend.
   */
  function fillVerticalGradient(frame, top, bottom) {
    throw MS.todo("fillVerticalGradient");
  }

  /**
   * Fill the whole frame with an elliptical radial gradient, like CSS
   * `radial-gradient(rx ry at cx cy, inner, outer)`.
   * For each pixel center (x + 0.5, y + 0.5):
   *   d = sqrt( ((x + 0.5 − cx) / rx)² + ((y + 0.5 − cy) / ry)² )
   *   t = min(d, 1)
   * and write mixColor(inner, outer, t). Overwrite, don't blend.
   */
  function fillRadialGradient(frame, cx, cy, rx, ry, inner, outer) {
    throw MS.todo("fillRadialGradient");
  }

  Object.assign(MS, { hex, blendPixel, fillRect, fillCircle, mixColor, fillVerticalGradient, fillRadialGradient });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("01")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: how many pixels does each rectangle cover?
const myGuess = null; // TODO: your guess as [first, second]
const count = f => { let n = 0; for (let i = 3; i < f.data.length; i += 4) if (f.data[i]) n++; return n; };
const a = MS.createFrame(5, 1), b = MS.createFrame(5, 1);
MS.fillRect(a, 0.4, 0, 2, 1, [255, 255, 255]);   // starts at x = 0.4
MS.fillRect(b, 0.6, 0, 2, 1, [255, 255, 255]);   // starts at x = 0.6
MS.reveal("pixels covered by each 2-wide rectangle", myGuess, [count(a), count(b)]);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: a circle of radius 5 has area πr² ≈ 78.5. How many pixels does it fill?
const myGuess = null; // TODO: your guess, a whole number
const f = MS.createFrame(21, 21);
MS.fillCircle(f, 10.5, 10.5, 5, [255, 255, 255]);
let n = 0;
for (let i = 3; i < f.data.length; i += 4) if (f.data[i]) n++;
MS.reveal("pixels in a radius-5 circle", myGuess, n);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: half-transparent red painted onto an empty, transparent pixel.
const myGuess = null; // TODO: your guess as [r, g, b, a]
const f = MS.createFrame(1, 1);
MS.blendPixel(f, 0, 0, [255, 0, 0, 128]);
MS.reveal("the pixel after blending", myGuess, MS.getPixel(f, 0, 0));
}


<details><summary>Answers</summary>

1. `[2, 2]`. Edges round to 0 → 2 and 1 → 3.
2. 81. Close to 78.5 but not equal: pixels are all-or-nothing, so the count wobbles around the true area.
3. `[255, 0, 0, 128]`. The color stays pure red; only the alpha says "half there".

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Forget the division



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: blend onto nothing, then layer the result over white.
// TODO(break-it), in src/01-shapes.js → blendPixel:
//   remove the "divide by outAlpha" step from the color math.
//   Keep the outAlpha math itself.
const f = MS.createFrame(1, 1);                  // transparent
MS.blendPixel(f, 0, 0, [255, 0, 0, 128]);        // 50% red onto nothing
const result = MS.getPixel(f, 0, 0);
console.log("50% red on nothing:", result);
const bg = MS.createFrame(1, 1);
MS.clearFrame(bg, [255, 255, 255]);              // white card
MS.blendPixel(bg, 0, 0, result);                 // put the result on top
console.log("...then over white:", MS.getPixel(bg, 0, 0));
// TODO: before the change the second line is a clean pink.
// After it, what color do you get, and why is it darker?
}


Which tests fail? Dark halos around cut-out images in real software are this exact bug.

#### 2. Make the right edge inclusive



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: two half-transparent rectangles that share an edge at x = 10.
// TODO(break-it), in src/01-shapes.js → fillRect:
//   make the rectangle also cover the pixel at x = right.
const f = MS.createFrame(20, 1);
MS.clearFrame(f, [255, 255, 255]);
MS.fillRect(f, 0, 0, 10, 1, [0, 0, 255, 128]);   // covers 0..9
MS.fillRect(f, 10, 0, 10, 1, [0, 0, 255, 128]);  // covers 10..19
console.log("x = 9: ", MS.getPixel(f, 9, 0));
console.log("x = 10:", MS.getPixel(f, 10, 0));
console.log("x = 11:", MS.getPixel(f, 11, 0));
// TODO: before the change all three match. After it, what happens at x = 10?
// Why would that look like a seam?
}


#### 3. Skip the bounding box



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: how long do 20 small circles take on a full-size frame?
// TODO(break-it), in src/01-shapes.js → fillCircle:
//   loop over every pixel in the frame instead of the circle's bounding box.
const f = MS.createFrame(1080, 1920);
const start = Date.now();
for (let i = 0; i < 20; i++) MS.fillCircle(f, 540, 960, 8, [255, 255, 255, 40]);
console.log("20 circles of radius 8:", Date.now() - start, "ms");
// TODO: run before and after. Roughly how many times slower is the full-frame loop?
// How many pixels does each version test per circle?
}


### Checkpoint

When all 17 tests pass, the checkpoint unlocks (layer 00 explains how it works). Its traps: squares that share edges, see-through ink painted onto nothing, then everything composited over a gradient. The three questions are below.



In [ ]:
// Checkpoint 01 · Shapes & blending. Replace each null with your answer.
MS.answers("01", {
  // 1. You blend [255, 255, 255, 128] (half-transparent white) over an opaque black pixel.
  //    What is the pixel afterwards? Answer as [r, g, b, a].
  q1: null,
  // 2. A pixel is [0, 0, 255, 128]. You blend [255, 0, 0, 64] over it.
  //    What is its new alpha (0–255)?
  q2: null,
  // 3. How many pixels does fillRect(frame, 1.5, 0.5, 3, 2, color) cover?
  //    (Check what Math.round does with exact halves.)
  q3: null,
});

In [ ]:
await checkpoint("01")


### In the real world

- Canvas's [`globalAlpha`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/globalAlpha) and CSS [`opacity`](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/opacity) both use Porter–Duff "over"; canvas exposes the full set of operators as [`globalCompositeOperation`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/globalCompositeOperation).
- Browsers usually keep pixels premultiplied internally. That's why a canvas pixel with very low alpha can come back from [`getImageData`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/getImageData) with slightly different colors than you put in: precision was lost in the multiply and divide.
- Motion Studio's `KineticIntro.tsx` paints its background with the CSS gradient above, then adds a second radial glow and a dark vignette on top. Those are three layers blended "over" each other, which you can now do by hand.

### Going further (optional)

- **Read the original.** [Alvy Ray Smith, "Alpha and the History of Digital Compositing" (1995)](http://alvyray.com/Memos/CG/Microsoft/7_alpha.pdf): how the alpha channel was invented, by one of its inventors.
- **Antialiased circles.** Instead of in-or-out, use coverage `clamp(r − distance + 0.5, 0, 1)` and multiply it into the color's alpha before blending. Compare the edges at scale in the test pictures.
- **Premultiplied mode.** Write `blendPremultiplied` where colors are stored already multiplied by alpha. Show that the formula no longer needs a division, and find which test values it can't reproduce exactly after 8-bit rounding.


## Layer 02 · Glyphs

_A font is a lookup table from characters to tiny pictures._

**You'll write:** `drawGlyph`, `measureText`, `drawText` in `src/02-glyphs.js`
**Builds on:** layers 00–01 (especially `fillRect`)

KineticIntro is mostly words: **DESIGN.**, **ANIMATE.**, **RENDER.**, then **MOTION·STUDIO**. In this layer you'll draw text using nothing but the rectangles you already have and a font made of `#` and `.` characters. In layer 08 you'll switch to the browser's real fonts, and this layer is how you'll understand what the browser is doing for you.

### Under the hood

#### Two kinds of fonts

**[Bitmap fonts](https://en.wikipedia.org/wiki/Computer_font#Bitmap_fonts)** store each character as a grid of on/off pixels. Arcade machines, early terminals and the first Macintosh used them; [Susan Kare](https://en.wikipedia.org/wiki/Susan_Kare) drew the original Mac fonts on graph paper in 1983–84. They're simple and fast, but they only look right at one size.

**Outline fonts** ([TrueType](https://developer.apple.com/fonts/TrueType-Reference-Manual/), [OpenType](https://learn.microsoft.com/en-us/typography/opentype/spec/)) store each character as curves, so they can be drawn at any size. Drawing one means rasterizing the curves into pixels with antialiasing, which is the "which pixels does this shape cover?" problem from layer 01, done very well. That's what the browser does when you write `font-size: 150px`.

You're building a bitmap font renderer because it shows the whole pipeline with no magic: _character → lookup → grid → pixels_.

> **Study buddy:** "Explain how outline fonts get rasterized, connecting it to the pixel-center rule I learned in layer 01." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Explain%20how%20outline%20fonts%20get%20rasterized%2C%20connecting%20it%20to%20the%20pixel-center%20rule%20I%20learned%20in%20layer%2001.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Explain%20how%20outline%20fonts%20get%20rasterized%2C%20connecting%20it%20to%20the%20pixel-center%20rule%20I%20learned%20in%20layer%2001.)

#### The font file

`lib/font5x7.js` is the font. Every glyph is 7 rows of 5 cells:

```text
   MS.FONT_5x7.N  =               drawn at scale 3:
   [ "#...#",                     ■■■         ■■■
     "##..#",                     ■■■■■■      ■■■
     "#.#.#",                     ■■■   ■■■   ■■■
     "#..##",          ──►        ■■■      ■■■■■■
     "#...#",                     ■■■         ■■■
     "#...#",                     ■■■         ■■■
     "#...#" ]                    ■■■         ■■■
                                  (each # becomes a 3 × 3 block)
```

Reading it is a nested loop: for each row `r`, for each column `c`, if the cell is `#`, fill a `scale × scale` square at `(x + c × scale, y + r × scale)`. Scaling by repeating each pixel as a block like this is called **[nearest-neighbor](https://en.wikipedia.org/wiki/Nearest-neighbor_interpolation)** scaling. It's why pixel art stays crisp.

#### Laying out a line

Text layout is the job of deciding where each glyph goes. Ours is the simplest possible version:

```text
   x                                                    one glyph = 5 cells
   │◄─ 5 ─►│◄1►│◄─ 5 ─►│◄1►│◄─ 5 ─►│                    tracking  = gap between glyphs
   ┌───────┐   ┌───────┐   ┌───────┐
   │   H   │   │   I   │   │   !   │                    width = 3 × 5 + 2 × 1 = 17 cells
   └───────┘   └───────┘   └───────┘                          (× scale for pixels)
   ◄──────────── measureText = 17 ───────────►
```

- **Advance** is how far you move right after each glyph: its width plus the tracking.
- **Tracking** is uniform extra space *between* glyphs, so none after the last one (CSS calls it [`letter-spacing`](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/letter-spacing)). [Kerning](https://en.wikipedia.org/wiki/Kerning), which tightens specific pairs like "AV", is different; our font skips it.

**Alignment needs measuring first.** To center text on `x`, you need its total width before drawing the first glyph, so the left edge is `x − width / 2`. Real layout engines work in the same two passes: measure everything, then place everything. KineticIntro centers every word, so you'll use this constantly.

> **Study buddy:** "Ask me questions until I can explain the difference between advance, tracking and kerning. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20the%20difference%20between%20advance%2C%20tracking%20and%20kerning.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20the%20difference%20between%20advance%2C%20tracking%20and%20kerning.%20No%20code.)

#### Characters are trickier than they look

- **Case.** The font has only uppercase. Mapping `"g"` to `"G"` (with [`toUpperCase`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/String/toUpperCase)) is a tiny case of what real fonts do with **fallback**: when the font lacks a character, find a substitute.
- **Missing characters.** When there's no substitute, show **tofu**: an empty box. (Google's font family [Noto](https://fonts.google.com/noto) is named for "no more tofu".) Tofu is ugly on purpose. It tells you something is missing instead of silently skipping it.
- **What counts as one character?** `"MOTION·STUDIO"` has 13 characters, and `·` is one of them. But try `"👋".length` in the console: it says 2. JavaScript strings count 16-bit units ([`String.length`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/String/length)), and characters beyond the first 65,536 take two, called a [surrogate pair](https://www.unicode.org/faq/utf_bom.html) in Unicode's UTF-16 encoding. [`for...of`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Statements/for...of) and [spread](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/Spread_syntax) (`[...text]`) walk real characters. A plain `for (let i = 0; i < text.length; i++)` splits that emoji in half. Pick your loop carefully.

#### Sizing for the real video

KineticIntro sets its words at a font size of 150 px in a 1080 px wide frame. Font size is the height of the font's whole design box (the "em"), not of a capital letter ([`font-size`](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/font-size)). In Inter, the font it uses, capitals are about 73% of that, so roughly 110 px tall. Our capitals are 7 cells tall, so a scale of about 16 matches. Can **ANIMATE.** fit across 1080 pixels at scale 16 with tracking 1? How large can the scale get before it doesn't? Work it out with `measureText` once it passes.

### Your turn

1. **measureText.** Start here: it's pure arithmetic and has the clearest tests. What should it return for the empty string? For one glyph?
2. **drawGlyph.** Uppercase first, and tofu if it's still missing. The loop is in *The font file* above.
3. **drawText.** Measure once, work out the starting x from `align`, then walk the characters, advancing by one glyph plus tracking each time.

Once it works, the test pictures show **DESIGN.** in big pixel letters, plus an alignment test with a pink center line and one tofu box.

> **Study buddy:** "I'll paste my measureText. Don't fix it; give me one input where it's wrong and let me work out why." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27ll%20paste%20my%20measureText.%20Don%27t%20fix%20it%3B%20give%20me%20one%20input%20where%20it%27s%20wrong%20and%20let%20me%20work%20out%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27ll%20paste%20my%20measureText.%20Don%27t%20fix%20it%3B%20give%20me%20one%20input%20where%20it%27s%20wrong%20and%20let%20me%20work%20out%20why.)



### ✏️ Your code for layer 02

This cell is your `src/02-glyphs.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2002%20%28Glyphs%29%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 02 · Glyphs — text from a hand-made bitmap font.
// Read the chapter first (docs/02-glyphs.md). Each comment below is a spec; the tests check it.
//
// The font lives in lib/font5x7.js as MS.FONT_5x7: an object mapping a
// character to 7 strings of 5 characters, "#" for ink and "." for empty.
// Draw with fillRect from layer 01.
(function (MS) {
  const GLYPH_W = 5;
  const GLYPH_H = 7;

  /**
   * Draw one character with its top-left corner at (x, y).
   * Each "#" cell becomes a filled square of size `scale` × `scale`.
   * - The font only has uppercase letters: "a" must draw like "A".
   * - A character missing from the font draws "tofu": a hollow 5 × 7 box
   *   (the outer ring of cells filled, the inside empty).
   */
  function drawGlyph(frame, ch, x, y, scale, color) {
    throw MS.todo("drawGlyph");
  }

  /**
   * How wide is `text` in pixels?
   * Each glyph is GLYPH_W cells wide, and `tracking` empty cells sit
   * BETWEEN glyphs (not after the last one). Multiply cells by `scale`.
   *   measureText("AB", 1)    → 5 + 1 + 5 = 11
   *   measureText("AB", 3)    → 33
   *   measureText("", 3)      → 0
   */
  function measureText(text, scale, tracking = 1) {
    throw MS.todo("measureText");
  }

  /**
   * Draw a line of text and return its width (same number measureText gives).
   * opts.tracking : empty cells between glyphs (default 1)
   * opts.align    : "left" (default) — x is the left edge
   *                 "center"         — x is the middle
   *                 "right"          — x is the right edge
   * A space draws nothing but still takes up a glyph's width.
   */
  function drawText(frame, text, x, y, scale, color, opts = {}) {
    throw MS.todo("drawText");
  }

  Object.assign(MS, { GLYPH_W, GLYPH_H, drawGlyph, measureText, drawText });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("02")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: how wide is "DESIGN." at scale 4 with tracking 1?
const myGuess = null; // TODO: your guess in pixels
MS.reveal('measureText("DESIGN.", 4)', myGuess, MS.measureText("DESIGN.", 4));
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: "Ä" isn't in the font. How many of its 35 cells get ink?
const myGuess = null; // TODO: a number from 0 to 35
const f = MS.createFrame(5, 7);
MS.drawGlyph(f, "Ä", 0, 0, 1, [255, 255, 255]);
let n = 0;
for (let i = 3; i < f.data.length; i += 4) if (f.data[i]) n++;
MS.reveal("cells inked for Ä", myGuess, n);
console.log(MS.asciiFrame(f));
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: centered on x = 20, "II" is 11 wide, so its left edge is 14.5.
// Which column gets the first ink?
const myGuess = null; // TODO: a column number
const f = MS.createFrame(40, 7);
MS.drawText(f, "II", 20, 0, 1, [255, 255, 255], { align: "center" });
let first = -1;
for (let x = 0; x < f.width; x++) if (MS.getPixel(f, x, 0)[3]) { first = x; break; }
MS.reveal("first inked column", myGuess, first);
}


<details><summary>Answers</summary>

1. 164 pixels: 7 glyphs × 5 cells × 4, plus 6 gaps × 1 × 4.
2. 20. "Ä" isn't in the font and uppercasing it doesn't help, so you get tofu: the 20 cells around the edge of the box.
3. Column 15. Your `fillRect` rounds edges, and `round(14.5)` is 15. Text lands on whole pixels, just like rectangles.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Tracking after every glyph



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: is centered text still centered?
// TODO(break-it), in src/02-glyphs.js → measureText:
//   count tracking after every glyph, including the last one.
const f = MS.createFrame(40, 7);
MS.drawText(f, "II", 20, 0, 1, [255, 255, 255], { align: "center" });
let left = -1, right = -1;
for (let x = 0; x < f.width; x++) if (MS.getPixel(f, x, 0)[3]) { if (left < 0) left = x; right = x; }
console.log("empty columns on the left:", left, "| on the right:", f.width - 1 - right);
console.log(MS.asciiFrame(f));
// TODO: before the change the two sides differ by one column (from rounding).
// After it, which way does the text shift, and by how much? Why that direction?
}


Real UI text that looks "slightly off-center" is often this bug.

#### 2. Loop by index



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: one emoji, drawn by your drawText.
// TODO(break-it), in src/02-glyphs.js → drawText:
//   walk the text with for (let i = 0; i < text.length; i++) and text[i].
console.log('"HI 👋".length =', "HI 👋".length, '  [..."HI 👋"].length =', [..."HI 👋"].length);
const f = MS.createFrame(30, 7);
MS.drawText(f, "HI 👋", 0, 0, 1, [255, 255, 255]);
console.log(MS.asciiFrame(f));
// TODO: count the tofu boxes before and after the change. Why does one emoji become two?
}


#### 3. Skip the uppercase step



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: mixed-case text.
// TODO(break-it), in src/02-glyphs.js → drawGlyph:
//   look the character up exactly as given, without uppercasing it.
const f = MS.createFrame(36, 7);
MS.drawText(f, "Design", 0, 0, 1, [255, 255, 255]);
console.log(MS.asciiFrame(f));
// TODO: predict what you'll see after the change, then run. Which letters survive?
}


### Checkpoint

When all 11 tests pass, the checkpoint unlocks (layer 00 explains how it works). Its traps: lowercase text, an emoji, middle dots, and three lines that must all be centered. The three questions are below.



In [ ]:
// Checkpoint 02 · Glyphs. Replace each null with your answer.
MS.answers("02", {
  // 1. How wide, in pixels, is "HELLO, WORLD!" at scale 2 with tracking 1?
  q1: null,
  // 2. How many tofu boxes does drawText draw for "Ça va? 👋"?
  q2: null,
  // 3. drawText(frame, "HI", 100, 0, 3, color, { align: "center" }):
  //    which pixel column gets the first ink?
  q3: null,
});

In [ ]:
await checkpoint("02")


### In the real world

- Canvas's [`measureText`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/measureText) and [`textAlign`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/textAlign) do what you just did, for outline fonts, with kerning and shaping on top.
- Text shaping engines like [HarfBuzz](https://harfbuzz.github.io/) handle what our font ignores: kerning, ligatures, scripts that join letters, and right-to-left text.

### Going further (optional)

- **Design your own glyphs.** Add lowercase letters or symbols to a copy of the font. What's the smallest grid where lowercase "g" still has a readable descender?
- **Outlines.** Draw text with a 1-cell dark outline by drawing it shifted in 8 directions, then on top. Count how many `fillRect` calls that costs, and think about why real renderers don't do it this way.


## Layer 03 · Time

_A video is a function from frame numbers to pictures._

**You'll write:** `secondsToFrames`, `framesToSeconds`, `timecode`, `createComposition`, `renderFrame`, `frameAt` in `src/03-time.js`
**Builds on:** layers 00–02

Everything so far draws one picture. KineticIntro is 180 of them, shown 30 a second for six seconds. This layer adds time, though not the way you might expect: instead of nudging things a little on every tick, you'll describe each frame as a function of its number, `n`. That one decision, which [Remotion](https://www.remotion.dev/docs/the-fundamentals) is built on, is what lets a video jump to any frame, render in any order, and come out the same every time.

### Under the hood

#### Motion is still pictures, fast

Show still pictures quickly enough and your eye reads them as movement. Film shows 24 a second, most web video 30 or 60. That rate is the **frame rate**, in frames per second (fps). Frame `n` is on screen from `n / fps` seconds until the next frame starts:

```text
  time (ms)   0          33.3        66.7        100
              │ frame 0   │ frame 1   │ frame 2   │ frame 3 …     at 30 fps
              └─ 1000 ÷ 30 = 33.3 ms ─┘
```

So converting is multiplying or dividing by fps: 1.5 seconds at 30 fps is 45 frames, and 45 frames is 1.5 seconds. Only one direction needs rounding, because a clip can't last half a frame.

#### frame = f(n)

There are two ways to make a ball move. One keeps state and changes it on every tick: `x = x + 2`. The other computes the picture straight from the frame number: `x = 2 × n`.

```text
  by mutation:   tick → x += 2 → draw      frame 40 depends on frames 0–39 having run, in order
  by f(n):       n    → x = 2n → draw      frame 40 depends only on 40
```

The second way wins everywhere that matters. A timeline can jump to frame 140 without playing 0–139. Frames can render out of order, or in parallel: Remotion splits a long video across many CPU cores or cloud machines. And the same `n` always gives the same picture, so a frame can be tested like any other function.

A **composition** packages that function with what a player needs to know: `width`, `height`, `fps`, `durationInFrames`, and `render(frame, n)`. Those are the same props as Remotion's [`<Composition>`](https://www.remotion.dev/docs/composition). KineticIntro's are 1080 × 1920, 30 fps, 180 frames. `renderFrame` gives `render` a brand-new, empty frame every time, so a frame can't depend on what was drawn before it.

> **Study buddy:** "Ask me questions until I can explain why describing frame n as a function of n beats moving things a little on every tick. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2003%20%28Time%29%3A%20frame%20%3D%20f%28n%29%20and%20a%20player%20that%20keeps%20time.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20describing%20frame%20n%20as%20a%20function%20of%20n%20beats%20moving%20things%20a%20little%20on%20every%20tick.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2003%20%28Time%29%3A%20frame%20%3D%20f%28n%29%20and%20a%20player%20that%20keeps%20time.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20describing%20frame%20n%20as%20a%20function%20of%20n%20beats%20moving%20things%20a%20little%20on%20every%20tick.%20No%20code.)

#### Timecode

Video editors label frames with [SMPTE timecode](https://en.wikipedia.org/wiki/SMPTE_timecode): `HH:MM:SS:FF`, hours, minutes, seconds, then the frame within the current second. The last field counts up to `fps − 1`, not 99. At 24 fps, frame 100 is 4 seconds and 4 frames: `00:00:04:04`.

#### A player keeps time

To play a composition, a browser page asks for [`requestAnimationFrame`](https://developer.mozilla.org/en-US/docs/Web/API/Window/requestAnimationFrame) callbacks. The browser calls back before each screen refresh, 60 or 120 times a second depending on the screen, with a timestamp. The player turns the time that has passed into a frame number:

```text
  screen refreshes (60 Hz):   │  │  │  │  │  │  │  │  │      every 16.7 ms
  video frames (30 fps):      │  0  │  1  │  2  │  3  │      every 33.3 ms
                                → each frame is shown on two refreshes
```

A frame stays on screen until the next one has *started*, so the player counts whole frames and never rounds up. If drawing a frame takes too long, the player doesn't fall behind: on the next refresh it shows whatever frame the clock says, skipping the ones in between. That's a **dropped frame**. An animation built on `x += 2` would slow down instead. Past the end, a looping player starts again from 0; the [remainder operator](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/Remainder) `%` does exactly that wrap-around.

From this layer on, the test pictures move. The page plays them with `lib/present.js`, which uses *your* `frameAt` and `renderFrame` to decide what to show.

### Your turn

Work through `src/03-time.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **secondsToFrames, framesToSeconds.** Only one of them rounds. Which, and why?
2. **timecode.** How many frames are in one minute at a given fps? Which operator gives you what's left over after taking out whole seconds?
3. **createComposition.** What counts as a positive whole number? Layer 00's `createFrame` asked the same question.
4. **renderFrame.** Why make a new frame on every call instead of keeping one? What would frame 3 contain if you didn't?
5. **frameAt.** Frame `k` starts at `k × 1000 / fps` ms. At 40 ms into a 30 fps clip, which frames have started? What should a 90-frame loop show at exactly 3000 ms?

Run the tests after each function. A failing test tells you exactly what it expected.

Once it works, the test pictures play: a ball that crosses the frame in exactly one second, and a clock showing every frame's timecode.

> **Study buddy:** "I'll paste my frameAt. Don't fix it. Give me an elapsed time where it would show the wrong frame, and let me work out why." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2003%20%28Time%29%3A%20frame%20%3D%20f%28n%29%20and%20a%20player%20that%20keeps%20time.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27ll%20paste%20my%20frameAt.%20Don%27t%20fix%20it.%20Give%20me%20an%20elapsed%20time%20where%20it%20would%20show%20the%20wrong%20frame%2C%20and%20let%20me%20work%20out%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2003%20%28Time%29%3A%20frame%20%3D%20f%28n%29%20and%20a%20player%20that%20keeps%20time.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27ll%20paste%20my%20frameAt.%20Don%27t%20fix%20it.%20Give%20me%20an%20elapsed%20time%20where%20it%20would%20show%20the%20wrong%20frame%2C%20and%20let%20me%20work%20out%20why.)



### ✏️ Your code for layer 03

This cell is your `src/03-time.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2003%20%28Time%29%3A%20frame%20%3D%20f%28n%29%20and%20a%20player%20that%20keeps%20time.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2003%20%28Time%29%3A%20frame%20%3D%20f%28n%29%20and%20a%20player%20that%20keeps%20time.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 03 · Time — frame = f(n) and a player that keeps time.
// Read the chapter first (docs/03-time.md). Each comment below is a spec; the tests check it.
// From here on, `frame` is a picture (layer 00) and `n` is a frame number: 0, 1, 2, …
(function (MS) {
  /**
   * How many frames does `seconds` last at `fps` frames per second?
   * Round to the nearest whole frame (Math.round).
   *   secondsToFrames(2, 30) → 60
   */
  function secondsToFrames(seconds, fps) {
    throw MS.todo("secondsToFrames");
  }

  /**
   * How many seconds do `n` frames last at `fps`? Don't round.
   *   framesToSeconds(45, 30) → 1.5
   */
  function framesToSeconds(n, fps) {
    throw MS.todo("framesToSeconds");
  }

  /**
   * Frame number n as a timecode string "HH:MM:SS:FF": hours, minutes,
   * seconds, and the frame within the current second. Every field has
   * two digits. fps is a whole number.
   *   timecode(30 * 61 + 5, 30) → "00:01:01:05"
   */
  function timecode(n, fps) {
    throw MS.todo("timecode");
  }

  /**
   * Check and return a composition: a video described as a function.
   *   spec = { width, height, fps, durationInFrames, render }
   * width, height, fps and durationInFrames must be positive whole numbers,
   * and render a function. Throw an Error otherwise. Return a new object
   * with exactly those five properties.
   * render(frame, n) draws frame number n into `frame`.
   */
  function createComposition(spec) {
    throw MS.todo("createComposition");
  }

  /**
   * Draw frame number n of composition `comp` and return it:
   * make a new frame of the composition's size (createFrame), call
   * comp.render(frame, n), and return the frame.
   * Throw an Error unless n is a whole number from 0 to durationInFrames − 1.
   */
  function renderFrame(comp, n) {
    throw MS.todo("renderFrame");
  }

  /**
   * Which frame should a player show after `elapsedMs` milliseconds?
   * Count the frames that have fully started: frame k starts at
   * k × 1000 / fps ms. Past the last frame:
   *   loop = true   start again from 0 (a 90-frame clip shows frame 0 after frame 89)
   *   loop = false  stay on the last frame
   * elapsedMs is never negative.
   */
  function frameAt(elapsedMs, fps, durationInFrames, loop = true) {
    throw MS.todo("frameAt");
  }

  Object.assign(MS, { secondsToFrames, framesToSeconds, timecode, createComposition, renderFrame, frameAt });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("03")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: how long is one frame on screen at 30 fps, in milliseconds?
const myGuess = null; // TODO: your guess, rounded to 2 decimals
MS.reveal("one frame at 30 fps, in ms", myGuess, Math.round(MS.framesToSeconds(1, 30) * 100000) / 100);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: after exactly one second, which frame does a 30 fps player show?
const myGuess = null; // TODO: your guess, a frame number
MS.reveal("frame on screen at 1000 ms", myGuess, MS.frameAt(1000, 30, 90));
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: a 120 Hz screen refreshes every 8.33 ms. During one second of
// a 30 fps video, on how many of those 120 refreshes does the picture change?
const myGuess = null; // TODO: your guess, a whole number
let changes = 0, last = -1;
for (let k = 0; k < 120; k++) {
  const n = MS.frameAt((k * 1000) / 120, 30, 300);
  if (n !== last) { changes++; last = n; }
}
MS.reveal("refreshes that show a new frame", myGuess, changes);
}


<details><summary>Answers</summary>

1. 33.33 ms (1000 ÷ 30). That's the whole time budget for drawing a frame if you want to play in real time.
2. 30, not 29 or 31. Frames are numbered from 0, so after one second frames 0–29 have finished and frame 30 has just started, the same way pixel coordinates start at 0.
3. 30. The screen refreshes 120 times, but the video only has 30 new pictures each second, so each one stays up for 4 refreshes in a row. The video looks the same on a 60 Hz screen; each frame just gets 2 refreshes.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Reuse the frame



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: does each frame start clean?
// TODO(break-it), in src/03-time.js → renderFrame:
//   make the frame once, keep it in a variable outside the function,
//   and reuse it on every call. Change nothing else.
const comp = MS.createComposition({
  width: 12, height: 3, fps: 30, durationInFrames: 12,
  render(frame, n) { MS.setPixel(frame, n, 1, [255, 255, 255]); },   // one dot per frame
});
console.log(MS.asciiFrame(MS.renderFrame(comp, 3)));
console.log(MS.asciiFrame(MS.renderFrame(comp, 7)));
// TODO: before the change each frame has one dot. After it, what does frame 7 show, and why?
}


Players render frames out of order all the time: scrubbing, seeking, and rendering in parallel. A frame that depends on the one before it breaks all three.

#### 2. Round instead of counting



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: which frame is on screen at 20 ms?
// TODO(break-it), in src/03-time.js → frameAt:
//   round to the nearest frame (Math.round) instead of counting started frames.
for (const ms of [0, 10, 20, 30, 40, 50]) console.log(ms + " ms → frame " + MS.frameAt(ms, 30, 90));
// TODO: after the change, at which times does the player show a frame that hasn't started yet?
}


#### 3. Forget to loop



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: what happens at the end of the clip?
// TODO(break-it), in src/03-time.js → frameAt:
//   return the number of started frames without looping or holding.
const comp = MS.createComposition({ width: 4, height: 4, fps: 30, durationInFrames: 90, render() {} });
for (const ms of [2900, 2990, 3000, 3100]) {
  const n = MS.frameAt(ms, 30, comp.durationInFrames);
  try { MS.renderFrame(comp, n); console.log(ms + " ms → frame " + n + ": drawn"); }
  catch (e) { console.log(ms + " ms → frame " + n + ": " + e.message); }
}
// TODO: after the change, what happens at 3 seconds, and which function notices?
}


### Checkpoint

When all 13 tests pass, the checkpoint unlocks (layer 00 explains how it works). Its traps: moments a hair before a frame starts, a clip that loops, a clip that holds its last frame, half a frame to round, and a timecode caption. The three questions are below.



In [ ]:
// Checkpoint 03 · Time. Replace each null with your answer.
MS.answers("03", {
  // 1. At 24 fps, how many frames long is a 2.5-second clip?
  q1: null,
  // 2. A 30 fps player loops a 60-frame clip. 2,350 ms after it starts,
  //    which frame number is on screen?
  q2: null,
  // 3. What is the timecode of frame 4,000 at 25 fps? Answer as a string.
  q3: null,
});

In [ ]:
await checkpoint("03")


### In the real world

- In Remotion, [`useCurrentFrame()`](https://www.remotion.dev/docs/use-current-frame) hands every component its `n`, and a component is your `render`: a function of the frame number.
- For real video files, a `<video>` element's [`requestVideoFrameCallback`](https://developer.mozilla.org/en-US/docs/Web/API/HTMLVideoElement/requestVideoFrameCallback) tells you exactly which decoded frame is on screen, the same question `frameAt` answers.
- US broadcast video runs at 29.97 fps, a leftover from adding color to black-and-white TV, so its timecode has to skip some labels to stay in step with the clock: [drop-frame timecode](https://en.wikipedia.org/wiki/SMPTE_timecode#Drop-frame_timecode).

### Going further (optional)

- **Read the original.** The HTML Standard's [animation frames](https://html.spec.whatwg.org/multipage/imagebitmap-and-animations.html#animation-frames) section: when `requestAnimationFrame` callbacks run, and what the timestamp means.
- Give the player controls: pause, scrub to any frame (easy, because a frame is just `f(n)`), and play at half speed.
- Write `timecodeDropFrame(n)` for 29.97 fps, which skips the labels `:00` and `:01` at the start of every minute except every tenth minute. Check it against an online timecode calculator.


## Layer 04 · Interpolate

_Every moving number is a line between keyframes._

**You'll write:** `lerp`, `inverseLerp`, `clamp`, `interpolate`, `interpolateColors` in `src/04-interpolate.js`
**Builds on:** layers 00–03

Layer 03 gave every frame a number. Now each frame needs to know where things are: how faded a word is at frame 12, how far the light sweep has traveled by frame 150. KineticIntro answers every one of those questions with a single function from Remotion, `interpolate`. Here's its light sweep:

```js
interpolate(frame, [140, 172], [-20, 130], { extrapolateLeft: "clamp", extrapolateRight: "clamp" })
```

Between frames 140 and 172, the sweep travels from −20% to 130% of the way across the logo, and before and after that it stays put. By the end of this layer you'll have written `interpolate` yourself, options and all, plus a version for colors.

### Under the hood

#### Keyframes and in-betweens

In hand-drawn animation, a lead animator drew the **keyframes**, the poses that matter, and assistants called [in-betweeners](https://en.wikipedia.org/wiki/Inbetweening) drew every frame between them. Software took over that job, now called *tweening*: you say where something is at a few moments, and the computer fills in the rest.

The simplest way to fill in is a straight line. **lerp**, short for *linear interpolation*, walks a fraction `t` of the way from `a` to `b`:

```text
  lerp(a, b, t) = a + (b − a) × t

  t:   0          0.25       0.5                   1          1.5
       40 ─────── 50 ─────── 60 ────────────────── 80 · · · · 100
       a                                           b
```

`t` is a fraction, not a frame: 0 is the start, 1 is the end, 0.5 is halfway. Nothing stops it going past 1 or below 0; the line just keeps going. You've already written one: layer 01's `mixColor` is a lerp on four channels at once.

#### Map, don't move

`inverseLerp` runs lerp backwards: given a value, which `t` gives it? Put the two together and you can map any range onto any other. Say a word fades in between frames 30 and 50, and you want its opacity at frame 35:

```text
  frames    30 ────●─────────── 50      inverseLerp(30, 50, 35) = 0.25
                  35
  opacity    0 ────●───────────  1      lerp(0, 1, 0.25) = 0.25
```

First turn the input into a fraction of its range, then turn that fraction into a point in the output range. For two keyframes, that's all `interpolate` does. Notice what's missing: nothing is stored between frames. The opacity at frame 35 comes from 35 alone, which is exactly what layer 03's frame = f(n) asked for.

> **Study buddy:** "Ask me questions until I can explain why interpolate is inverseLerp followed by lerp, using a fade-out as the example. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2004%20%28Interpolate%29%3A%20lerp%2C%20clamp%2C%20ranges%2C%20extrapolation.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20interpolate%20is%20inverseLerp%20followed%20by%20lerp%2C%20using%20a%20fade-out%20as%20the%20example.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2004%20%28Interpolate%29%3A%20lerp%2C%20clamp%2C%20ranges%2C%20extrapolation.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20interpolate%20is%20inverseLerp%20followed%20by%20lerp%2C%20using%20a%20fade-out%20as%20the%20example.%20No%20code.)

#### Many keyframes

With more than two keyframes, the ranges become a chain of **segments**. Find the segment the input falls in, then map within that segment alone:

```text
  inputRange    0 ──────────── 30 ────── 50 ──────────────── 90
  outputRange   0              100       60                  60
                └──── seg 0 ────┘└ seg 1 ─┘└────── seg 2 ─────┘

  input 40 falls in segment 1:   t = (40 − 30) / (50 − 30) = 0.5
                                 output = lerp(100, 60, 0.5) = 80
```

Keyframes don't have to be evenly spaced, and in real animation they rarely are: each word in KineticIntro takes 26 frames to come in and 22 to leave. What `interpolate` does insist on is that the inputs are **strictly increasing**. If they weren't, one input could sit inside two segments at once, and there'd be no single right answer. The outputs can go up, down, or both.

#### Past the ends

Before the first keyframe and after the last, `interpolate` has to make something up. That's **extrapolation**. Say a word scales up from 0.8 to 1 between frames 0 and 10 (KineticIntro's words start at 80% size). What's its scale at frame 20?

```text
  scale
   1.2 ┤                    ·      "extend" keeps going along the end segment's line
   1.1 ┤               ·
   1.0 ┤          ●━━━━━━━━━━━━    "clamp" holds the end value
   0.9 ┤     ·
   0.8 ┤●                          ● a keyframe
       └┬─────────┬─────────┬──▶ frame
        0        10        20
```

- **"extend"**, Remotion's default, keeps going along the end segment's line: 1.2, and still growing. With several keyframes, "the end segment" means the **first** segment before the first keyframe and the **last** one after the last keyframe.
- **"clamp"** holds the end value: 1.
- **"identity"** returns the input itself: 20. It only makes sense when input and output measure the same thing.

`extrapolateLeft` and `extrapolateRight` choose separately for each end. "extend" is the honest mathematical continuation, but anything with a natural limit, like opacity, a progress bar or a fade, needs "clamp". KineticIntro clamps every single call.

#### Easing plugs in at t

One more option, `easing`, is a function that reshapes `t` before it's mapped to the output. It takes the progress through the segment and returns a new progress:

```text
  t → t         ●────●────●────●────●────●────●────●────●      even steps
  t → t × t     ●●─●──●───●─────●──────●───────●────────●      slow start, fast finish
```

Each dot is where the value is after another eighth of the time. Because easing works on `t`, the start and end don't move: `t` = 0 still gives the first output and `t` = 1 the last. Easing only changes *when* the value gets there. With several keyframes, every segment gets its own trip from 0 to 1, so the easing starts over at each keyframe. Past the ends, with "extend", `t` goes below 0 or above 1, and the easing is applied to it there too: with `t × t`, the extension curves instead of running straight. Layer 05 builds real easing curves; this layer just calls whatever function it's given.

#### Colors are four numbers

`interpolateColors` blends colors one channel at a time: red with red, green with green, blue with blue, alpha with alpha, each its own interpolation over the same input range. Two things differ from plain numbers:

- **It always clamps.** There's no color past "fully red" to extend into; the channel would leave 0–255.
- **It rounds.** A channel is a whole byte. A `Uint8ClampedArray` would round for you, but it sends exact halves to the nearest *even* number (layer 00's bonus). Rounding with [`Math.round`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Math/round) yourself sends them up instead: 128.5 becomes 129, where a clamped array would give 128.

Remotion's own [`interpolateColors()`](https://www.remotion.dev/docs/interpolate-colors) works the same way: channel by channel, clamped at both ends, rounded.

### Your turn

Work through `src/04-interpolate.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **lerp.** What should `lerp(a, b, 0)` and `lerp(a, b, 1)` give? Check your formula against both before running anything.
2. **inverseLerp.** If `lerp(a, b, t)` gives `v`, what's `t`? Why does `a === b` have to be an error?
3. **clamp.** What should `clamp(1.3, 0, 1)` and `clamp(−0.2, 0, 1)` give?
4. **interpolate.** Check the ranges first; the spec lists what to reject. Then, on paper, for an inputRange of `[0, 4, 6]`: which segment do the inputs −1, 0, 4, 5 and 9 use? Last, for each option: does it act on the input, on `t`, or on the output?
5. **interpolateColors.** What's the same about interpolating a color and interpolating a number, and what are the two differences? Is there anything you've already written that you can lean on?

Run the tests after each function. A failing test tells you exactly what it expected.

Once it works, the test pictures move: two blocks that show "extend" against "clamp", a dot that visits four corners, and a color ramp through three stops.

> **Study buddy:** "Give me an inputRange with uneven keyframes and five inputs, some before and after the ends. Ask me which segment each one uses and what t is. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2004%20%28Interpolate%29%3A%20lerp%2C%20clamp%2C%20ranges%2C%20extrapolation.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20an%20inputRange%20with%20uneven%20keyframes%20and%20five%20inputs%2C%20some%20before%20and%20after%20the%20ends.%20Ask%20me%20which%20segment%20each%20one%20uses%20and%20what%20t%20is.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2004%20%28Interpolate%29%3A%20lerp%2C%20clamp%2C%20ranges%2C%20extrapolation.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20an%20inputRange%20with%20uneven%20keyframes%20and%20five%20inputs%2C%20some%20before%20and%20after%20the%20ends.%20Ask%20me%20which%20segment%20each%20one%20uses%20and%20what%20t%20is.%20No%20code.)



### ✏️ Your code for layer 04

This cell is your `src/04-interpolate.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2004%20%28Interpolate%29%3A%20lerp%2C%20clamp%2C%20ranges%2C%20extrapolation.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2004%20%28Interpolate%29%3A%20lerp%2C%20clamp%2C%20ranges%2C%20extrapolation.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 04 · Interpolate — lerp, clamp, ranges, extrapolation.
// Read the chapter first (docs/04-interpolate.md). Each comment below is a spec; the tests check it.
(function (MS) {
  /**
   * Linear interpolation: a at t = 0, b at t = 1, and everything along the
   * straight line between and beyond. Don't clamp t.
   *   lerp(10, 20, 0.25) → 12.5
   */
  function lerp(a, b, t) {
    throw MS.todo("lerp");
  }

  /**
   * The opposite of lerp: what t gives v? (v − a) as a fraction of (b − a).
   *   inverseLerp(10, 20, 15) → 0.5
   * Throw an Error if a === b.
   */
  function inverseLerp(a, b, v) {
    throw MS.todo("inverseLerp");
  }

  /**
   * v, kept between min and max.
   *   clamp(12, 0, 10) → 10
   */
  function clamp(v, min, max) {
    throw MS.todo("clamp");
  }

  /**
   * Map `input` from inputRange onto outputRange, like Remotion's interpolate.
   *   interpolate(5, [0, 10], [0, 100]) → 50
   * Both ranges have the same length (at least 2), and inputRange must be
   * strictly increasing; throw an Error otherwise. With more than two points,
   * use the segment the input falls in. Past the ends, options decide:
   *   extrapolateLeft / extrapolateRight:
   *     "extend"   (default) keep going along the end segment's line
   *     "clamp"    stay at the end value
   *     "identity" return the input itself
   *   easing: a function applied to t, the progress through the segment
   *     (0 at its start, 1 at its end), before t is mapped to the output.
   *     With "extend", t keeps going past 0 and 1 and is still eased.
   *     Default: t => t.
   */
  function interpolate(input, inputRange, outputRange, options = {}) {
    throw MS.todo("interpolate");
  }

  /**
   * Interpolate between colors ([r, g, b] or [r, g, b, a]; missing alpha is
   * 255), one channel at a time, clamped at both ends. Round each channel to
   * a whole number and return [r, g, b, a].
   *   interpolateColors(0.5, [0, 1], [[0, 0, 0], [200, 200, 200]]) → [100, 100, 100, 255]
   */
  function interpolateColors(input, inputRange, colors) {
    throw MS.todo("interpolateColors");
  }

  Object.assign(MS, { lerp, inverseLerp, clamp, interpolate, interpolateColors });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("04")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: a fade-in from frame 0 to 30, with only its right end clamped.
// What opacity does it give at frame −10, before the fade starts?
const myGuess = null; // TODO: your guess, to 2 decimals
MS.reveal("opacity at frame −10", myGuess, Math.round(MS.interpolate(-10, [0, 30], [0, 1], { extrapolateRight: "clamp" }) * 100) / 100);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: what color is halfway between pure red and pure green?
const myGuess = null; // TODO: your guess, as [r, g, b, a]
MS.reveal("halfway from red to green", myGuess, MS.interpolateColors(0.5, [0, 1], [[255, 0, 0], [0, 255, 0]]));
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: a move from 0 to 100 over frames 0–20, eased with t => t * t.
// At which whole frame does it first reach halfway (50)?
const myGuess = null; // TODO: your guess, a frame number
let n = 0;
while (MS.interpolate(n, [0, 20], [0, 100], { easing: t => t * t }) < 50) n++;
MS.reveal("first frame at or past 50", myGuess, n);
}


<details><summary>Answers</summary>

1. −0.33. Only the right end was clamped, so the left end still extends, backwards along the fade's line to an opacity below zero. This happens for real: KineticIntro shows each word from two frames *before* its animation starts, and those frames sit before the first keyframe. A browser treats a negative opacity as 0, so you'd never notice here, but a position or a scale has no such safety net. Clamp both ends.
2. `[128, 128, 0, 255]`: a dim olive, not the bright yellow you might expect. Each channel is only halfway up, and sRGB numbers aren't proportional to light: a channel at 128 gives off only about a fifth of the light of 255 (see the [sRGB transfer function](https://en.wikipedia.org/wiki/SRGB#Transfer_function_%28%22gamma%22%29)). Halfway in numbers is much darker than halfway in light. That's why CSS lets you choose where colors get mixed: [`color-mix(in oklab, red, lime)`](https://developer.mozilla.org/en-US/docs/Web/CSS/color_value/color-mix) blends in a space built to match how bright colors look.
3. 15. With `t × t`, the value reaches halfway when `t × t` = 0.5, so `t` ≈ 0.71: frame 14.1 of 20. Frame 15 is the first whole frame past it. An eased move can spend most of its time on the first half of the distance; layer 05 makes curves that ease in, out, or both.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Only ever use the first segment



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: what is finding the segment for?
// TODO(break-it), in src/04-interpolate.js → interpolate:
//   always use the first two keyframes, whatever the input. Change nothing else.
const keys = [0, 15, 30, 45, 60];
for (const n of [0, 10, 20, 35, 50]) {
  const x = MS.interpolate(n, keys, [10, 54, 54, 10, 10]), y = MS.interpolate(n, keys, [10, 10, 30, 30, 10]);
  console.log("frame " + n + " → (" + Math.round(x) + ", " + Math.round(y) + ")");
}
// TODO: the dot should visit four corners of a 64 × 40 frame. After the change, where does it go, and why?
}


#### 2. Ease the output instead of t



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: why does easing work on t?
// TODO(break-it), in src/04-interpolate.js → interpolate:
//   apply easing to the final output value instead of to t.
const easeIn = t => t * t;
for (const n of [0, 5, 10]) console.log("frame " + n + " → " + MS.interpolate(n, [0, 10], [100, 200], { easing: easeIn }));
// TODO: before the change the value goes 100, 125, 200. What does it do after, and why?
}


An easing function expects a progress between 0 and 1. Handed a value like 150, it has nothing sensible to say.

#### 3. Let any inputRange through



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: why must the inputs be strictly increasing?
// TODO(break-it), in src/04-interpolate.js → interpolate:
//   delete the check that the inputs are strictly increasing.
const keys = [0, 20, 10, 30];   // 10 comes after 20
for (const n of [5, 15, 25]) {
  try { console.log(n + " → " + MS.interpolate(n, keys, [0, 100, 200, 300])); }
  catch (e) { console.log(n + " → " + e.message); }
}
// TODO: 15 lies inside all three segments. Which one did your code use?
//   Is either answer right?
}


### Checkpoint

When all 14 tests pass, the checkpoint unlocks (layer 00 explains how it works). The picture is a long-exposure photo of a light moved by your `interpolate`. Its traps: keyframes that aren't evenly spaced, a light switched on before its first keyframe, another left on past both ends of a long chain of keyframes, easing that restarts every segment, and a fade in alpha. The three questions are below.



In [ ]:
// Checkpoint 04 · Interpolate. Replace each null with your answer.
MS.answers("04", {
  // 1. KineticIntro's light sweep is
  //      interpolate(frame, [140, 172], [-20, 130], { extrapolateLeft: "clamp", extrapolateRight: "clamp" })
  //    Where is the sweep at frame 164?
  q1: null,
  // 2. A title slides in with interpolate(frame, [0, 20], [-160, 0]) and no options.
  //    Where is it at frame 26?
  q2: null,
  // 3. A background fades through three colors:
  //      interpolateColors(frame, [0, 60, 90], [MS.hex("#1A2740"), MS.hex("#5B8CFF"), MS.hex("#FFFFFF")])
  //    What is its red channel at frame 70?
  q3: null,
});

In [ ]:
await checkpoint("04")


### In the real world

- Remotion's [`interpolate()`](https://www.remotion.dev/docs/interpolate) is this function at heart: same arguments, same "extend" default, the same "clamp" and "identity", plus a fourth extrapolation, "wrap". Newer versions add more on top: a different easing for each segment, `posterize` to update only every few frames, and outputs that aren't plain numbers, like CSS transform strings.
- CSS animations are keyframes on a 0–100% input range. An [`@keyframes`](https://developer.mozilla.org/en-US/docs/Web/CSS/@keyframes) rule lists values at percentages, the browser interpolates every frame between them, and [`animation-timing-function`](https://developer.mozilla.org/en-US/docs/Web/CSS/animation-timing-function) eases each pair of keyframes separately, the way your easing restarts every segment.
- GPUs have lerp built in: GLSL's [`mix()`](https://registry.khronos.org/OpenGL-Refpages/gl4/html/mix.xhtml) is lerp, and [`smoothstep()`](https://registry.khronos.org/OpenGL-Refpages/gl4/html/smoothstep.xhtml) is a clamped `inverseLerp` with easing built in.

### Going further (optional)

- **Read the original.** Remotion's [`interpolate.ts`](https://github.com/remotion-dev/remotion/blob/main/packages/core/src/interpolate.ts). The file has grown to handle strings and more, but look for `findRange` and `interpolateFunction`: a few dozen lines between them that do what yours does. Does its `findRange` pick the same segment as yours when an input lands exactly on a keyframe? Does it matter?
- Add Remotion's fourth mode, "wrap": past the end, start the range over, like layer 03's looping player.
- Mix light instead of numbers: convert each channel to linear light with the sRGB transfer function, interpolate, convert back, and compare your red-to-green midpoint with `interpolateColors`.


## Layer 05 · Easing

_A curve that decides when, not where._

**You'll write:** `bezierPoint`, `bezierSlope`, `solveBezierX`, `cubicBezier` in `src/05-easing.js`
**Builds on:** layers 00–04

Layer 04's `interpolate` takes an `easing` function and reshapes `t` with it. This layer makes the functions worth passing. KineticIntro defines two curves in its code: **APPLE**, `bezier(0.16, 1, 0.3, 1)`, which every entrance uses, and **SMOOTH**, `bezier(0.6, 0, 0.2, 1)`, for every exit. Those four numbers are the same ones CSS takes in `cubic-bezier()`. By the end of this layer you'll turn any four numbers into an easing function, the way a browser does, with a little numerical root-finding.

### Under the hood

#### Why ease

Nothing in the real world starts or stops instantly. A ball rolling to a stop slows down; a car pulling away speeds up. Linear motion jumps to full speed on the first frame and halts dead on the last, and our eyes read that as mechanical. An **easing function** fixes this by changing *when* the value gets where it's going: it takes the fraction of the time that has passed, `x`, and returns the fraction of the distance covered, `y`. It starts at (0, 0) and ends at (1, 1), and the shape between is the personality.

CSS describes that shape with a [cubic Bézier curve](https://www.w3.org/TR/css-easing-1/#cubic-bezier-easing-functions) and two control points, P1 = (x1, y1) and P2 = (x2, y2). Its keywords are just named curves: `ease` is `cubic-bezier(0.25, 0.1, 0.25, 1)`, `ease-in` is `(0.42, 0, 1, 1)`, `ease-out` is `(0, 0, 0.58, 1)`.

#### A Bézier curve is lerps of lerps

In 1959, Paul de Casteljau, a mathematician at Citroën, found a way to describe smooth car-body curves to a computer; Pierre Bézier at Renault published the same curves a few years later, and they took his name. De Casteljau's way of computing a point on the curve uses nothing but layer 04's `lerp`. For an easing curve, the four control values along one axis are 0, `p1`, `p2` and 1:

```text
  control values:   0           p1           p2           1
                          ●            ●            ●             each ● is u of the way from
                                 ●            ●                   the value on its left to the
                                      B(u)                        value on its right, one row up
```

Run it once with the x values and once with the y values, and you get the point (x, y) on the curve at parameter `u`. Multiply the lerps out and you get a cubic polynomial in `u` instead, which is how browsers write it. Either way works. `bezierSlope` is that polynomial's derivative: how fast the curve moves along one axis as `u` grows.

#### The catch: u isn't time

As `u` runs from 0 to 1, the point travels along the curve, but `x` and `y` *both* depend on `u`. An easing function is asked a different question: at time `x`, what's `y`? To answer it, first find the `u` where the curve's x equals the time you were given, then read the curve's y at that `u`:

```text
  x (time) ──solve──▶ u ──bezierPoint(y1, y2, u)──▶ y (progress)
```

There's no tidy formula for that first arrow. (Cubic equations do have a formula, but it's fussy and loses precision.) So browsers find `u` numerically.

#### Newton's method

Guess a `u`, see how far its x is from the target, then use the slope to jump to where a straight line would hit the target: `u ← u − error ÷ slope`. Here's ease-in (x1 = 0.42, x2 = 1) looking for x = 0.6, starting from the guess `u` = 0.6:

```text
  step   u          x at u       error        slope   next u = u − error ÷ slope
  0      0.6        0.76896      +0.169       1.037   0.4370
  1      0.4370     0.58058      −0.0194      1.256   0.4525
  2      0.4525     0.59988      −0.00012     1.240   0.452605
  3      0.452605   0.60000000   −0.000000005    close enough
```

The error goes 0.17, 0.02, 0.0001, 0.000000005: each step roughly doubles the number of correct digits. That's [Newton's method](https://en.wikipedia.org/wiki/Newton%27s_method), and it's usually done in three or four steps.

> **Study buddy:** "Ask me questions until I can explain why Newton's method divides by the slope, and what goes wrong when the slope is nearly zero. Use a curve that isn't an easing curve. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2005%20%28Easing%29%3A%20a%20cubic-B%C3%A9zier%20solver.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20Newton%27s%20method%20divides%20by%20the%20slope%2C%20and%20what%20goes%20wrong%20when%20the%20slope%20is%20nearly%20zero.%20Use%20a%20curve%20that%20isn%27t%20an%20easing%20curve.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2005%20%28Easing%29%3A%20a%20cubic-B%C3%A9zier%20solver.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20Newton%27s%20method%20divides%20by%20the%20slope%2C%20and%20what%20goes%20wrong%20when%20the%20slope%20is%20nearly%20zero.%20Use%20a%20curve%20that%20isn%27t%20an%20easing%20curve.%20No%20code.)

#### Bisection, the safety net

Newton's method has a weak spot: where the curve goes flat, the slope is near zero, and dividing by it flings `u` far away. **Bisection** can't fail. The curve's x only ever increases, so the answer is always inside a known gap, starting with 0 to 1. Check the middle: if its x is too big, the answer is in the left half; otherwise it's in the right half. Each check halves the gap:

```text
  [0 ─────────────────────────────── 1]    x at 0.5 too big?   keep the left half
  [0 ─────────────── 0.5]                  x at 0.25 too small? keep the right half
          [0.25 ──── 0.5]                  … until the x at the middle is within 1e-7
```

The gap is in `u`, but the spec's 1e-7 is about `x`, and an easing curve's x can change up to 3 times faster than `u`, so it can take about 25 halvings. It's slow but sure, so real solvers use both. [WebKit's `UnitBezier`](https://github.com/WebKit/WebKit/blob/main/Source/WebCore/platform/graphics/UnitBezier.h), the code behind Safari's CSS animations, takes a first guess from a table of 11 samples, tries up to 4 Newton steps, and bisects if they don't get close enough.

#### Only x has to behave

CSS insists that x1 and x2 stay between 0 and 1, and it rejects the curve otherwise. Outside that range the curve's x can turn around and run backwards, which would mean time running backwards: one moment `x` would have three different `u`s and no single answer. That's the same reason layer 04 insisted on increasing keyframes.

The y values have no such rule. A y control point above 1 makes the value **overshoot** its target and settle back, like `cubic-bezier(0.34, 1.56, 0.64, 1)`; one below 0 makes it back up before it starts, called **anticipation**. Both are old tricks from hand-drawn animation.

### Your turn

Work through `src/05-easing.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **bezierPoint.** What should it give at `u` = 0 and `u` = 1, whatever `p1` and `p2` are? Work the picture above by hand for `p1` = 0.1, `p2` = 0.7 and `u` = 0.5 before you run anything.
2. **bezierSlope.** If you multiplied the lerps out into a polynomial, how do you take its derivative? The test "bezierSlope is how fast bezierPoint changes" lets you check your answer numerically.
3. **solveBezierX.** What's a good first guess for `u`? When should you stop trusting Newton's method? If you use bisection, what tells you which half to keep?
4. **cubicBezier.** It returns a function. What does that function do with x = −0.5? Where do x1 and x2 get checked: once, or on every call?

Run the tests after each function. A failing test tells you exactly what it expected.

Once it works, the test pictures show five easing curves with their control points, five dots making the same move with five timings, and a word rising in with APPLE and fading out with SMOOTH.

> **Study buddy:** "I'll describe how my solveBezierX works, in words. Don't fix it. Give me a curve and an x where it would fail, and let me work out why." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2005%20%28Easing%29%3A%20a%20cubic-B%C3%A9zier%20solver.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27ll%20describe%20how%20my%20solveBezierX%20works%2C%20in%20words.%20Don%27t%20fix%20it.%20Give%20me%20a%20curve%20and%20an%20x%20where%20it%20would%20fail%2C%20and%20let%20me%20work%20out%20why.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2005%20%28Easing%29%3A%20a%20cubic-B%C3%A9zier%20solver.%20Here%27s%20what%20I%20want%20help%20with%3A%20I%27ll%20describe%20how%20my%20solveBezierX%20works%2C%20in%20words.%20Don%27t%20fix%20it.%20Give%20me%20a%20curve%20and%20an%20x%20where%20it%20would%20fail%2C%20and%20let%20me%20work%20out%20why.)



### ✏️ Your code for layer 05

This cell is your `src/05-easing.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2005%20%28Easing%29%3A%20a%20cubic-B%C3%A9zier%20solver.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2005%20%28Easing%29%3A%20a%20cubic-B%C3%A9zier%20solver.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 05 · Easing — a cubic-Bézier solver.
// Read the chapter first (docs/05-easing.md). Each comment below is a spec; the tests check it.
(function (MS) {
  /**
   * One coordinate of a cubic Bézier easing curve at parameter u (0 to 1).
   * The curve's four control values are 0, p1, p2, 1: it starts at 0, ends
   * at 1, and p1 and p2 pull it in between.
   *   bezierPoint(0.2, 0.8, 0.25) → 0.2125
   */
  function bezierPoint(p1, p2, u) {
    throw MS.todo("bezierPoint");
  }

  /**
   * How fast bezierPoint(p1, p2, u) changes as u grows: its derivative.
   *   bezierSlope(0.3, 0.9, 0) → 0.9    (3 × p1 at the start)
   *   bezierSlope(1, 0, 0.5)   → 0      (this curve goes flat in the middle)
   */
  function bezierSlope(p1, p2, u) {
    throw MS.todo("bezierSlope");
  }

  /**
   * Find the u (0 to 1) where the curve's x reaches `x`, so that
   *   |bezierPoint(x1, x2, u) − x| < 1e-7
   * x1 and x2 are between 0 and 1, so there is exactly one such u.
   * x = 0 gives 0 and x = 1 gives 1. It must work for every curve,
   * including ones that go flat (slope 0) somewhere.
   */
  function solveBezierX(x1, x2, x) {
    throw MS.todo("solveBezierX");
  }

  /**
   * CSS's cubic-bezier(x1, y1, x2, y2): return an easing function that
   * takes a time progress x and returns the curve's y at that x.
   * - Throw an Error if x1 or x2 is outside 0–1 (y1 and y2 can be anything).
   * - The function clamps x to 0–1 first, and returns exactly 0 at x = 0
   *   and exactly 1 at x = 1.
   *   cubicBezier(0.42, 0, 1, 1)(0.5) → about 0.3154   (CSS ease-in)
   */
  function cubicBezier(x1, y1, x2, y2) {
    throw MS.todo("cubicBezier");
  }

  Object.assign(MS, { bezierPoint, bezierSlope, solveBezierX, cubicBezier });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("05")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: CSS ease-in is cubic-bezier(0.42, 0, 1, 1). Halfway along its curve
// (u = 0.5), what fraction of the time has passed? That's the curve's x at u = 0.5.
const myGuess = null; // TODO: your guess, to 4 decimals
MS.reveal("ease-in's x at u = 0.5", myGuess, MS.bezierPoint(0.42, 1, 0.5));
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: cubic-bezier(0.34, 1.56, 0.64, 1) overshoots. What's the highest
// value it reaches? Its control point says 1.56.
const myGuess = null; // TODO: your guess, to 2 decimals
const back = MS.cubicBezier(0.34, 1.56, 0.64, 1);
let highest = 0;
for (let i = 0; i <= 1000; i++) highest = Math.max(highest, back(i / 1000));
MS.reveal("the highest value", myGuess, Math.round(highest * 100) / 100);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: KineticIntro's APPLE, cubic-bezier(0.16, 1, 0.3, 1). What fraction
// of the time does it take to cover 90% of the distance?
const myGuess = null; // TODO: your guess, to 2 decimals
const APPLE = MS.cubicBezier(0.16, 1, 0.3, 1);
let x = 0;
while (APPLE(x) < 0.9) x = Math.round((x + 0.01) * 100) / 100;
MS.reveal("time to cover 90%", myGuess, x);
}


<details><summary>Answers</summary>

1. 0.6575. Halfway along the curve, two thirds of the time has already gone: `u` is the curve's own parameter, not time. That gap is why `solveBezierX` exists.
2. 1.1. A control point pulls the curve toward it but never reaches it, so 1.56 only lifts the peak to about 1.1. The same is true of every Bézier handle, which is why designers drag handles further than the shape they want.
3. 0.33. APPLE covers 90% of the move in a third of the time and spends the other two thirds on the last 10%. That long, gentle settle is what makes KineticIntro's words feel like they land rather than stop.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Skip the solve



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: why solve for u at all?
// TODO(break-it), in src/05-easing.js → cubicBezier:
//   skip solveBezierX and use x itself as u. Change nothing else.
const easeIn = MS.cubicBezier(0.42, 0, 1, 1);
for (const x of [0.25, 0.5, 0.75]) console.log("ease-in at x = " + x + " → " + easeIn(x).toFixed(4));
// TODO: what does ease-in give at x = 0.5 after the change? Which curve are you really drawing now?
}


#### 2. Newton's method with no safety net



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: what is the fallback for?
// TODO(break-it), in src/05-easing.js → solveBezierX:
//   use Newton's method only: start at u = x, take 8 steps, return u. Nothing else.
for (const x of [0.3, 0.45, 0.5, 0.55]) {
  const u = MS.solveBezierX(1, 0, x);
  console.log("x = " + x + " → u = " + u + ", the curve's x there: " + MS.bezierPoint(1, 0, u));
}
// TODO: this curve goes flat at u = 0.5. Which x values go wrong after the change, and why those?
}


#### 3. Settle for less precision



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: how accurate is accurate enough?
// TODO(break-it), in src/05-easing.js → solveBezierX:
//   stop as soon as the curve's x is within 0.01 of x, instead of 1e-7.
const APPLE = MS.cubicBezier(0.16, 1, 0.3, 1);
for (const x of [0.05, 0.1, 0.2, 0.4]) console.log("x = " + x + " → a 1920-pixel move is at " + (APPLE(x) * 1920).toFixed(1) + " px");
// TODO: how many pixels does the looser answer move things? Would you see it on a 1080 × 1920 video?
}


A steep curve magnifies small errors in x: APPLE starts out more than six times steeper than a straight line. The precision you need depends on how far things move on screen, which is why the spec asks for 1e-7.

### Checkpoint

When all 15 tests pass, the checkpoint unlocks (layer 00 explains how it works). The picture is a freeze-frame: dots flying in from scattered places, each on its own easing curve, caught at the instant when they should line up into a word. Its traps: everyday curves, curves that start flat, go flat in the middle or end flat, curves that overshoot, and times outside 0 to 1. The three questions are below.



In [ ]:
// Checkpoint 05 · Easing. Replace each null with your answer.
MS.answers("05", {
  // 1. CSS's default timing function, ease, is cubic-bezier(0.25, 0.1, 0.25, 1).
  //    Halfway through the time (x = 0.5), how far along is the value? Round to 3 decimals.
  q1: null,
  // 2. KineticIntro brings each word in over 26 frames with APPLE, cubic-bezier(0.16, 1, 0.3, 1).
  //    What is the progress after 5 frames? Round to 2 decimals.
  q2: null,
  // 3. KineticIntro's SMOOTH is cubic-bezier(0.6, 0, 0.2, 1). At what time x does it reach
  //    exactly halfway (y = 0.5)? Round to 3 decimals. There's a way to do this without a solver.
  q3: null,
});

In [ ]:
await checkpoint("05")


### In the real world

- CSS transitions and animations take these curves directly: `transition-timing-function: cubic-bezier(0.16, 1, 0.3, 1)`. The keywords `ease`, `ease-in` and the rest are fixed curves defined in the [CSS Easing Functions](https://www.w3.org/TR/css-easing-1/) spec.
- Remotion's [`Easing.bezier()`](https://www.remotion.dev/docs/easing) is the same idea, code borrowed from React Native: it starts from a table of 11 samples for a better first guess, then takes Newton steps or bisects.
- Design tools speak the same language: Figma's custom easing, After Effects' graph editor and Apple's [`CAMediaTimingFunction`](https://developer.apple.com/documentation/quartzcore/camediatimingfunction) all describe timing with two control points.

### Going further (optional)

- **Read the original.** The [cubic Bézier easing functions](https://www.w3.org/TR/css-easing-1/#cubic-bezier-easing-functions) section of CSS Easing Functions. Notice what browsers do with times below 0 or above 1: instead of clamping, they extend the curve along its tangent lines.
- Do the same: replace clamping with CSS's tangent-line rule, and check that `interpolate` with "extend" now keeps moving smoothly past the end.
- Count your solver's Newton steps for 1,000 values of x, then add an 11-sample lookup table for the first guess, like Remotion's, and count again.


## Layer 06 · Springs

_A spring doesn't know where it's going to be, only what happens next._

**You'll write:** `springForce`, `stepSpring`, `spring`, `dampingRatio`, `settleFrame` in `src/06-springs.js`
**Builds on:** layers 00–05

Easing curves are drawn by hand: someone picks four numbers and the motion follows the shape exactly. A spring is different. You describe a physical object, how stiff it is, how much friction it has and how heavy it is, and the motion is whatever physics says happens. That's why spring motion feels alive, and why [Remotion's `spring()`](https://www.remotion.dev/docs/spring) is one of its most-used functions. KineticIntro itself only uses easing curves; springs are the other half of Remotion's motion toolkit. This layer simulates one, step by step, and finds out how small the steps have to be.

### Under the hood

#### A spring is a rule, not a curve

Picture a weight on a spring, lying on a slippery table. Pull it and let go. Two forces act on it. The spring pulls it back toward where it wants to rest, harder the further away it is: that's [Hooke's law](https://en.wikipedia.org/wiki/Hooke%27s_law). Friction, called **damping**, pushes against the motion, harder the faster it moves:

```text
  force = −stiffness × (position − target)  −  damping × velocity
          └──── the spring pulls it back ──┘   └─ friction slows it ─┘

  acceleration = force ÷ mass
```

That's the whole description, and it only says what happens *next*: given where the weight is and how fast it's moving, here's how its speed is changing. To find out where it is at frame 30, you have to play time forward from the start.

#### Stepping through time

Playing time forward means taking small steps of `dt` seconds. Each step uses the force to update the velocity, and the velocity to update the position. The order matters more than it looks:

```text
  explicit Euler:        position += velocity × dt      (the old velocity)
                         velocity += acceleration × dt

  semi-implicit Euler:   velocity += acceleration × dt
                         position += velocity × dt      (the new velocity)
```

Both look reasonable, and over one step they barely differ. Over thousands of steps, explicit Euler quietly adds energy every time: a spring with no friction swings a little wider each second, forever. [Semi-implicit Euler](https://en.wikipedia.org/wiki/Semi-implicit_Euler_method) keeps the swing the same size, which is why physics engines use it. It costs nothing extra; it's just the right order.

#### How small is small enough?

Each step pretends the force stays the same for `dt` seconds. That's fine while `dt` is short compared with how fast the spring swings. A spring's natural speed is `ω = √(stiffness ÷ mass)`, in radians per second, and semi-implicit Euler stays stable only while `ω × dt` is below 2. Past that, every step overshoots a little more than the last, and the numbers explode.

A video frame is far too long a step: at 30 fps, `dt` is 33 ms, so any spring stiffer than about 3,600 (with mass 1) blows up. This layer steps every **1 ms** instead, and frame `n` is 1000 × n ÷ fps milliseconds in, rounded to the nearest whole step. Notice what that costs: because frame = f(n), working out frame 90 means simulating all 3,000 steps from the start, every time.

> **Study buddy:** "Ask me questions until I can explain why updating the velocity before the position keeps a frictionless spring from gaining energy. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2006%20%28Springs%29%3A%20physics%20stepping%20and%20stability.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20updating%20the%20velocity%20before%20the%20position%20keeps%20a%20frictionless%20spring%20from%20gaining%20energy.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2006%20%28Springs%29%3A%20physics%20stepping%20and%20stability.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20updating%20the%20velocity%20before%20the%20position%20keeps%20a%20frictionless%20spring%20from%20gaining%20energy.%20No%20code.)

#### The damping ratio

Three numbers describe a spring, but its character comes down to one, the **damping ratio**:

```text
  ζ = damping ÷ (2 × √(stiffness × mass))

  ζ below 1   underdamped         overshoots, bounces, then settles
  ζ = 1       critically damped   arrives as fast as possible, never overshoots
  ζ above 1   overdamped          creeps in slowly, never overshoots
```

Remotion's default spring is stiffness 100, damping 10, mass 1, so ζ = 10 ÷ (2 × 10) = 0.5: a little bouncy.

#### When is it done?

A spring never quite arrives. Each bounce is smaller than the last, but it never reaches exactly zero. So "done" needs a definition: the spring has **settled** at the first frame from which it stays within some small distance of its target for good. Remotion's [`measureSpring()`](https://www.remotion.dev/docs/measure-spring) uses half a percent of the distance from `from` to `to`, and so does `settleFrame`. "Stays" matters: a bouncy spring can pass through the target zone on its way back out.

### Your turn

Work through `src/06-springs.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **springForce.** If the position is past the target, which way should the force point? How do you fill in config values that are missing without losing the ones you were given?
2. **stepSpring.** Which comes first, the new velocity or the new position? Why must the state you were given stay unchanged? (Think about frame = f(n).)
3. **spring.** How many 1 ms steps is frame 5 at 48 fps? What should frame 0 give, whatever the config?
4. **dampingRatio.** What makes ζ exactly 1?
5. **settleFrame.** If frame 20 is inside the zone but frame 25 is outside, can frame 20 be the answer? Do you need to call `spring()` for every frame, or can you keep stepping one simulation?

Run the tests after each function. A failing test tells you exactly what it expected.

Once it works, the test pictures show four springs, bouncy, default, critically damped and heavy, as graphs and as moving dots.

> **Study buddy:** "Give me three spring configs and ask me to rank them by how much they overshoot and how fast they settle, before I run anything. Then ask me to explain the ranking. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2006%20%28Springs%29%3A%20physics%20stepping%20and%20stability.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20three%20spring%20configs%20and%20ask%20me%20to%20rank%20them%20by%20how%20much%20they%20overshoot%20and%20how%20fast%20they%20settle%2C%20before%20I%20run%20anything.%20Then%20ask%20me%20to%20explain%20the%20ranking.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2006%20%28Springs%29%3A%20physics%20stepping%20and%20stability.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20three%20spring%20configs%20and%20ask%20me%20to%20rank%20them%20by%20how%20much%20they%20overshoot%20and%20how%20fast%20they%20settle%2C%20before%20I%20run%20anything.%20Then%20ask%20me%20to%20explain%20the%20ranking.%20No%20code.)



### ✏️ Your code for layer 06

This cell is your `src/06-springs.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2006%20%28Springs%29%3A%20physics%20stepping%20and%20stability.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2006%20%28Springs%29%3A%20physics%20stepping%20and%20stability.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 06 · Springs — physics stepping and stability.
// Read the chapter first (docs/06-springs.md). Each comment below is a spec; the tests check it.
(function (MS) {
  /**
   * The force a spring and its damper put on a mass at `position`, moving at
   * `velocity`, while the spring pulls it toward `target`:
   *   force = −stiffness × (position − target) − damping × velocity
   * config is { stiffness, damping, mass }. Any missing value (or a missing
   * config) takes Remotion's default: stiffness 100, damping 10, mass 1.
   */
  function springForce(position, velocity, target, config) {
    throw MS.todo("springForce");
  }

  /**
   * Move a spring forward by dt seconds. state is { position, velocity }.
   * Update the velocity first (acceleration = force ÷ mass), then move the
   * position using the NEW velocity. Return a new state object; don't change
   * the one passed in.
   *   stepSpring({ position: 1, velocity: 0 }, 0, { stiffness: 100, damping: 0, mass: 1 }, 0.01)
   *     → { position: 0.99, velocity: -1 }
   */
  function stepSpring(state, target, config, dt) {
    throw MS.todo("stepSpring");
  }

  /**
   * Like Remotion's spring(): where is a spring at `frame`, if it starts at
   * rest at `from` on frame 0 and is pulled toward `to`?
   * Simulate it with stepSpring in steps of 1 ms (dt = 0.001). Frame n is
   * Math.round(1000 × n ÷ fps) steps from the start. from defaults to 0 and
   * to to 1; config is passed on as it is.
   */
  function spring({ frame, fps, from = 0, to = 1, config } = {}) {
    throw MS.todo("spring");
  }

  /**
   * The damping ratio: damping ÷ (2 × √(stiffness × mass)), with the same
   * defaults. Below 1 the spring overshoots and bounces; from 1 up it doesn't.
   */
  function dampingRatio(config) {
    throw MS.todo("dampingRatio");
  }

  /**
   * The first frame from which spring() stays within threshold × |to − from|
   * of `to` (exactly on the edge counts as within) for every frame up to
   * 10 seconds (frame 10 × fps). Return Infinity if it's still outside at
   * 10 seconds.
   */
  function settleFrame({ fps, from = 0, to = 1, config, threshold = 0.005 } = {}) {
    throw MS.todo("settleFrame");
  }

  Object.assign(MS, { springForce, stepSpring, spring, dampingRatio, settleFrame });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("06")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: Remotion's default spring goes from 0 to 1. What's the highest value it reaches?
const myGuess = null; // TODO: your guess, to 2 decimals
let highest = 0;
for (let n = 0; n <= 60; n++) highest = Math.max(highest, MS.spring({ frame: n, fps: 30 }));
MS.reveal("the highest value", myGuess, Math.round(highest * 100) / 100);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: with damping 18 instead of 10, does the spring ever go past 1?
const myGuess = null; // TODO: true or false
let passes = false;
for (let n = 0; n <= 300; n++) if (MS.spring({ frame: n, fps: 30, config: { damping: 18 } }) > 1) passes = true;
MS.reveal("goes past 1", myGuess, passes);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: the default spring settles at frame 28 (30 fps). Make it four times
// heavier (mass 4) and keep everything else. When does it settle?
const myGuess = null; // TODO: your guess, a frame number
MS.reveal("settle frame with mass 4", myGuess, MS.settleFrame({ fps: 30, config: { mass: 4 } }));
}


<details><summary>Answers</summary>

1. 1.16. With ζ = 0.5, the default spring overshoots by 16% before settling back. There's a formula for the first overshoot, `e^(−πζ ÷ √(1 − ζ²))`, and it gives 0.163.
2. true, but only just: it peaks about a thousandth past 1. Damping 18 makes ζ = 0.9, and anything below 1 overshoots a little. Only from ζ = 1 up does a spring arrive without ever passing its target.
3. 123, more than four times as long. A heavier mass slows the swing (ω halves) *and* makes it bouncier (ζ halves to 0.25), so it takes longer to swing and more swings to settle.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Explicit Euler



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: does the order of the two updates matter?
// TODO(break-it), in src/06-springs.js → stepSpring:
//   move the position with the OLD velocity first, then update the velocity.
const config = { stiffness: 900, damping: 0, mass: 1 };   // no friction: it should swing forever, never growing
for (let s = 1; s <= 4; s++) {
  let far = 0;
  for (let n = (s - 1) * 30; n < s * 30; n++) far = Math.max(far, Math.abs(MS.spring({ frame: n, fps: 30, config }) - 1));
  console.log("second " + s + ": swings up to " + far.toFixed(3) + " from the target");
}
// TODO: before the change, the swing stays the same size. What happens after? Where does the energy come from?
}


#### 2. One step per frame



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: why 1 ms steps?
// TODO(break-it), in src/06-springs.js → spring:
//   take one step per frame (dt = 1 ÷ fps) instead of 1 ms steps.
const stiff = { stiffness: 4000, damping: 20, mass: 1 };
for (const n of [2, 4, 6, 8, 10]) console.log("frame " + n + ": stiff " + MS.spring({ frame: n, fps: 30, config: stiff }).toFixed(3) + ", default " + MS.spring({ frame: n, fps: 30 }).toFixed(3));
// TODO: which spring survives the change and which explodes? Work out ω × dt for each (ω = √(stiffness ÷ mass)).
}


#### 3. Ignore the frame rate



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: what is fps for?
// TODO(break-it), in src/06-springs.js → spring:
//   ignore fps and always use 30.
console.log("frame 15 at 30 fps: " + MS.spring({ frame: 15, fps: 30 }).toFixed(4));
console.log("frame 30 at 60 fps: " + MS.spring({ frame: 30, fps: 60 }).toFixed(4));
console.log("frame 12 at 24 fps: " + MS.spring({ frame: 12, fps: 24 }).toFixed(4));
// TODO: all three are half a second in. After the change, which disagree? How would the 60 fps video look?
}


### Checkpoint

When all 14 tests pass, the checkpoint unlocks (layer 00 explains how it works). The picture is another freeze-frame of flying dots, this time on springs. Each dot's position is worked out twice, in two ways that must agree, so if your two answers disagree even slightly, that letter scatters. Its traps: configs with some values missing, a spring that falls instead of rising, frames that aren't a whole number of milliseconds, other frame rates, and a settle frame that has to agree with your spring. The three questions are below.



In [ ]:
// Checkpoint 06 · Springs. Replace each null with your answer.
MS.answers("06", {
  // 1. A spring has stiffness 300 and mass 2. What damping makes it critically damped?
  //    Round to 2 decimals.
  q1: null,
  // 2. How many 1 ms steps does spring() simulate to work out frame 100 at 24 fps?
  q2: null,
  // 3. react-spring's default config is tension 170 and friction 26 (its names for
  //    stiffness and damping), with mass 1. What's its damping ratio, to 3 decimals?
  q3: null,
});

In [ ]:
await checkpoint("06")


### In the real world

- Remotion's [`spring()`](https://www.remotion.dev/docs/spring) takes the same config names with the same defaults. Instead of simulating in 1 ms steps, it uses the exact formula for a damped spring, one frame at a time. For bouncy springs it lands within about 0.005 of your answer; it treats overdamped springs as critically damped, so there it disagrees.
- Apple's SwiftUI describes springs by their *damping fraction*, which is the damping ratio, and CSS can fake a spring by listing many points in a [`linear()`](https://developer.mozilla.org/en-US/docs/Web/CSS/easing-function/linear) easing function.
- Game engines step their physics in small, fixed time steps for exactly the stability reasons in this chapter, whatever the screen's frame rate: see Glenn Fiedler's ["Fix Your Timestep!"](https://gafferongames.com/post/fix_your_timestep/).

### Going further (optional)

- **Read the original.** Glenn Fiedler's ["Integration Basics"](https://gafferongames.com/post/integration_basics/) compares explicit Euler, semi-implicit Euler and RK4 on a spring, with graphs.
- Replace the simulation with the exact formula for an underdamped spring, `to − (to − from) × e^(−ζωt) × (cos(ω_d t) + (ζω ÷ ω_d) × sin(ω_d t))` with `ω_d = ω√(1 − ζ²)`, and compare with your 1 ms answer at frame 10.
- Make `spring()` fast without breaking frame = f(n): cache states so a later frame can continue from an earlier one, and check you still get the same answer whatever order frames are asked in.


## Layer 07 · Transforms & blur

_A matrix is six numbers that say where everything goes._

**You'll write:** `translate`, `scale`, `rotate`, `multiply`, `transformPoint`, `invert`, `drawTransformed`, `boxBlur`, `blur` in `src/07-transforms.js`
**Builds on:** layers 00–06

Each word in KineticIntro arrives the same way: it grows from 80% to full size while it sharpens out of a 14-pixel blur, then keeps pushing slowly toward the camera before it blurs away again. In CSS that's two lines, `transform: scale(…)` and `filter: blur(…)`, and the browser does the work. In this layer you do the work: matrices that move, scale and turn anything, a way to draw a picture through a matrix without holes or dark edges, and a blur whose cost doesn't depend on how wide it is.

### Under the hood

#### Six numbers move everything

Moving, scaling, rotating and slanting are all the same kind of operation, an **affine transform**, and six numbers describe any of them. The [HTML canvas standard](https://html.spec.whatwg.org/multipage/canvas.html#transformations) and CSS's [`matrix()`](https://www.w3.org/TR/css-transforms-1/#funcdef-transform-matrix) both write them as `a, b, c, d, e, f`:

```text
  ┌ a  c  e ┐   ┌ x ┐        x' = a·x + c·y + e
  │ b  d  f │ × │ y │        y' = b·x + d·y + f
  └ 0  0  1 ┘   └ 1 ┘
```

The six numbers are easier to read as three arrows. `(e, f)` is where the origin `(0, 0)` lands. `(a, b)` is where one step along x, the point `(1, 0)`, lands, measured from there; `(c, d)` is the same for one step along y. Draw those three things and you've drawn the whole transform: every other point follows from them.

```text
  scale(2, 0.5)                       a slant: c = 0.5, the rest as in scale(1)
  (1, 0) → (2, 0)                     (1, 0) → (1, 0)
  (0, 1) → (0, 0.5)                   (0, 1) → (0.5, 1)

  ┌────┐                              ┌────┐          ┌────┐
  │    │    →    ┌────────┐           │    │    →      ╲    ╲
  │    │         └────────┘           │    │            ╲    ╲
  └────┘                              └────┘             └────┘
                                      (y points down, so the bottom moves right)
```

On a screen, y points **down**. A rotation that turns +x toward +y therefore looks clockwise, the opposite of the math textbook picture. Canvas and CSS both work this way.

#### Order matters

A transform made of steps is one matrix: **multiplying** two matrices gives a single matrix that does both. But which one happens first changes the result. Take the point `x = 3`:

```text
  scale by 2, then move right 4:    3 → 6 → 10
  move right 4, then scale by 2:    3 → 7 → 14
```

In `multiply(m, n)`, `n` happens first. That reads backwards at first, but it's the same order as function calls: in `m(n(p))`, n runs first.

Order is also how you scale around a point other than the origin. `scale(1.04)` on its own grows everything away from `(0, 0)`, the top-left corner, so a word in the middle of the screen would drift down and to the right. To grow around the middle, three steps: move the middle to the origin, scale, move it back.

#### Drawing backwards

To draw a picture through a matrix, the obvious plan is to take every source pixel, move it, and paint it where it lands. Try that with a 3× zoom: neighboring source pixels land 3 pixels apart, and the pixels in between are never painted.

```text
  forward: each source pixel goes somewhere     backward: each destination pixel asks
                                                "where did I come from?"
  █ █ █      →     █ · · █ · · █                █ █ █      ←     █ █ █ █ █ █ █ █ █
                   · · · · · · ·                                  (every pixel is filled)
                   holes
```

So renderers work **backwards**: for each pixel of the destination, find the point in the source that lands on it, and take the color from there. That needs the matrix that undoes `m`, its **inverse**. Not every matrix has one. The number `a·d − b·c` (the **determinant**) says how much the matrix scales areas. When it's 0, the matrix squashes everything onto a line or a point, and there's no way back: many source points land on the same spot.

#### Between the pixels

The point you find in the source is almost never exactly on a pixel. Pixel `(x, y)` is a little square from `(x, y)` to `(x + 1, y + 1)`, so its center is `(x + 0.5, y + 0.5)`. A point between centers takes a little of each of the four nearest pixels, more from the nearer ones. That's **bilinear sampling**: blend along x, then along y.

```text
     center (3.5, 0.5)       center (4.5, 0.5)
            ●─────────●──────────────────●
                      ↑ the sample, 30% of the way from left to right
                         gets 70% of the left pixel and 30% of the right
```

Nearest-neighbor sampling (just take the closest pixel) is faster, but edges turn into stairs as a picture rotates, and a slow move jumps a whole pixel at a time. Bilinear sampling lets a picture move by a fraction of a pixel, which matters for slow motion like KineticIntro's 4% push.

> **Study buddy:** "Ask me questions until I can explain why drawing through a matrix works backwards, from destination pixels to source points, and what the determinant has to do with it. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2007%20%28Transforms%20%26%20blur%29%3A%20matrices%2C%20resampling%20and%20blur.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20drawing%20through%20a%20matrix%20works%20backwards%2C%20from%20destination%20pixels%20to%20source%20points%2C%20and%20what%20the%20determinant%20has%20to%20do%20with%20it.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2007%20%28Transforms%20%26%20blur%29%3A%20matrices%2C%20resampling%20and%20blur.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20why%20drawing%20through%20a%20matrix%20works%20backwards%2C%20from%20destination%20pixels%20to%20source%20points%2C%20and%20what%20the%20determinant%20has%20to%20do%20with%20it.%20No%20code.)

#### Edges without dark fringes

Bilinear sampling at the edge of a picture mixes a pixel of the picture with a transparent pixel outside it. Here's the trap from layer 01. A transparent pixel is stored as `[0, 0, 0, 0]`: "black, invisible". Average a white pixel with it channel by channel, and you get `[128, 128, 128, 128]`, a half-transparent **grey**. The black that was never visible has leaked in, and every edge gets a dark fringe.

```text
  straight average                 premultiplied average
  [255,255,255,255]                color × alpha:  255·255 and 0·0
  [  0,  0,  0,  0]                alpha:          255 and 0
  ────────────────── ÷ 2           ─────────────────────────────
  [128,128,128,128]  grey          alpha 128, color (255·255) ÷ 255 = 255: still white
```

The fix is to average **premultiplied** colors: weight each pixel's color by its alpha, so a transparent pixel adds nothing to the color, only to the alpha. Divide by the total alpha at the end to get a plain color back. Browsers keep pixels premultiplied for exactly this reason.

#### A blur that doesn't care how big it is

The simplest blur is a **box blur**: each pixel becomes the average of the pixels in a box around it. A box of radius `r` is `2r + 1` pixels wide. Two tricks make it fast:

```text
  1. rows, then columns:  blurring every row, then every column, gives the same
                          result as averaging the whole square, but each pixel
                          adds up 2·(2r + 1) values instead of (2r + 1)².

  2. a running sum:       moving the box one pixel right changes only its two ends.
                          ┌───────────┐
                          │ a b c d e │ f        sum(b…f) = sum(a…e) − a + f
                          └───────────┘
                          Two operations per pixel, whether r is 1 or 100.
```

Pixels outside the frame count as transparent, so a blur fades a picture out at its edges instead of smearing the edge pixels outward. Average premultiplied, for the same reason as above.

#### Three boxes make a bell

A box blur looks boxy: a single bright dot becomes a flat square. Box-blur that square again and the profile becomes a triangle; a third time and it's a smooth bell, very close to a **Gaussian**. That's the [central limit theorem](https://en.wikipedia.org/wiki/Central_limit_theorem) at work: adding up a few random amounts quickly gives a bell curve.

```text
  one box        two boxes        three boxes
  ▁▁████▁▁       ▁▂▄██▄▂▁        ▁▂▅██▅▂▁
```

A Gaussian blur's size is its **standard deviation**, σ (sigma), in pixels, and that's what CSS's [`blur()`](https://www.w3.org/TR/filter-effects-1/#funcdef-filter-blur) takes: `blur(14px)` means σ = 14. To pick `r` for three boxes, match the spread. A box of radius `r` has a variance (σ²) of `r(r + 1) ÷ 3`, and variances add up when blurs are stacked, so three boxes give `r(r + 1)`. Setting that equal to σ² and solving gives the radius in `blur`'s spec. For σ = 5 it's `round((√101 − 1) ÷ 2)` = round(4.52) = 5.

### Your turn

Work through `src/07-transforms.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **translate and scale.** Using the three arrows, where do `(0, 0)`, `(1, 0)` and `(0, 1)` land for each? What should `scale(3)` do when there's no second number?
2. **rotate.** After a quarter turn, where should `(1, 0)` land? And `(0, 1)`? What about a turn by any angle: what are the coordinates of a point one step from the origin at that angle?
3. **transformPoint.** Which of the six numbers multiply x, which multiply y, and which are just added?
4. **multiply.** If `n` sends the origin to some point, where does `multiply(m, n)` send it? Answer the same question for `(1, 0)` and `(0, 1)`, and you have all six numbers.
5. **invert.** What undoes `scale(4)`? What undoes `translate(3, −1)`? For any matrix, you have two equations, `x' = …` and `y' = …`: can you solve them for x and y? Where does `a·d − b·c` turn up?
6. **drawTransformed.** Which point of `src` lands on the center of dest pixel `(2, 5)`? Which four src pixels surround that point, and how far is it from each of their centers? Do you need to visit every pixel of dest, or only the ones `src` can reach?
7. **boxBlur.** What do you keep in your running sums so a transparent pixel doesn't pull the color toward black? When the box slides by one pixel, which pixel leaves and which arrives?
8. **blur.** Once `boxBlur` works, this one is short. Check your radius for σ = 5 against the chapter.

Run the tests after each function. A failing test tells you exactly what it expected.

Once it works, the test pictures show one sprite drawn through five different matrices, the same word at four blur sizes, and a word that pushes in out of a blur, then spins away.

> **Study buddy:** "Give me two transforms, like 'rotate a quarter turn' and 'move right 5', and ask me where a point ends up in each order before I run anything. Keep going until I get the order of multiply right every time. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2007%20%28Transforms%20%26%20blur%29%3A%20matrices%2C%20resampling%20and%20blur.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20two%20transforms%2C%20like%20%27rotate%20a%20quarter%20turn%27%20and%20%27move%20right%205%27%2C%20and%20ask%20me%20where%20a%20point%20ends%20up%20in%20each%20order%20before%20I%20run%20anything.%20Keep%20going%20until%20I%20get%20the%20order%20of%20multiply%20right%20every%20time.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2007%20%28Transforms%20%26%20blur%29%3A%20matrices%2C%20resampling%20and%20blur.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20two%20transforms%2C%20like%20%27rotate%20a%20quarter%20turn%27%20and%20%27move%20right%205%27%2C%20and%20ask%20me%20where%20a%20point%20ends%20up%20in%20each%20order%20before%20I%20run%20anything.%20Keep%20going%20until%20I%20get%20the%20order%20of%20multiply%20right%20every%20time.%20No%20code.)



### ✏️ Your code for layer 07

This cell is your `src/07-transforms.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2007%20%28Transforms%20%26%20blur%29%3A%20matrices%2C%20resampling%20and%20blur.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2007%20%28Transforms%20%26%20blur%29%3A%20matrices%2C%20resampling%20and%20blur.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 07 · Transforms & blur — matrices, resampling and blur.
// Read the chapter first (docs/07-transforms.md). Each comment below is a spec; the tests check it.
(function (MS) {
  // A matrix is an array of six numbers [a, b, c, d, e, f], in the same order
  // as canvas's setTransform(a, b, c, d, e, f). It moves the point (x, y) to
  //   x' = a·x + c·y + e
  //   y' = b·x + d·y + f

  /** The matrix that moves every point by (tx, ty). */
  function translate(tx, ty) {
    throw MS.todo("translate");
  }

  /**
   * The matrix that scales x by sx and y by sy, around the origin (0, 0).
   * With one number, it scales both: scale(2) is scale(2, 2).
   */
  function scale(sx, sy) {
    throw MS.todo("scale");
  }

  /**
   * The matrix that rotates by `angle` radians around the origin. A positive
   * angle turns +x toward +y: on a screen, where y points down, that's clockwise.
   *   rotate(Math.PI / 2) moves (1, 0) to (0, 1).
   */
  function rotate(angle) {
    throw MS.todo("rotate");
  }

  /**
   * One matrix that does n first, then m: transforming a point with
   * multiply(m, n) gives the same point as transforming it with n, then with m.
   * Return a new array.
   */
  function multiply(m, n) {
    throw MS.todo("multiply");
  }

  /** Where matrix m moves the point (x, y), as [x', y']. */
  function transformPoint(m, x, y) {
    throw MS.todo("transformPoint");
  }

  /**
   * The matrix that undoes m: transformPoint(invert(m), ...) moves every point
   * m moved back where it came from. Some matrices squash every point onto a
   * line or a single point, like scale(0, 1); they can't be undone, so return
   * null for them (when a·d − b·c is 0).
   */
  function invert(m) {
    throw MS.todo("invert");
  }

  /**
   * Draw the frame `src` onto the frame `dest`, moved by the matrix m (which
   * maps src's coordinates to dest's), blending over what's already there
   * with your blendPixel.
   * - Pixel (x, y) covers the square from (x, y) to (x + 1, y + 1), so its
   *   center is (x + 0.5, y + 0.5). This is true in both frames.
   * - Each dest pixel gets the color of src at the point that m moves onto
   *   that pixel's center. Between pixel centers, blend the four nearest src
   *   pixels, weighted by how close their centers are (bilinear sampling).
   *   Pixels outside src count as fully transparent.
   * - Blend premultiplied: weight each src pixel's color by its alpha as well,
   *   so transparent pixels fade an edge without darkening it.
   * - Multiply the sampled alpha by opacity (default 1).
   * If m can't be inverted, draw nothing. Pixels of dest that src doesn't
   * reach must not change.
   */
  function drawTransformed(dest, src, m, opacity = 1) {
    throw MS.todo("drawTransformed");
  }

  /**
   * Blur a frame with a box of 2r + 1 pixels: first every row, then every
   * column. Each pixel becomes the average of the pixels in the box centered
   * on it. Pixels outside the frame count as fully transparent. Average
   * premultiplied, as in drawTransformed: alpha is the plain average, and the
   * color is weighted by alpha. r is a whole number.
   * Return a new frame, and leave `frame` unchanged. boxBlur(frame, 0)
   * returns an unchanged copy.
   */
  function boxBlur(frame, r) {
    throw MS.todo("boxBlur");
  }

  /**
   * A Gaussian blur with standard deviation sigma (in pixels), the way CSS's
   * filter: blur(sigma px) means it, made from three box blurs in a row,
   * each of radius r = Math.round((√(1 + 4σ²) − 1) ÷ 2).
   * Return a new frame, and leave `frame` unchanged.
   */
  function blur(frame, sigma) {
    throw MS.todo("blur");
  }

  Object.assign(MS, { translate, scale, rotate, multiply, transformPoint, invert, drawTransformed, boxBlur, blur });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("07")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: move right 10, then turn a quarter turn (clockwise on screen).
// Where does the origin (0, 0) end up?
const myGuess = null; // TODO: your guess, as [x, y]
const m = MS.multiply(MS.rotate(Math.PI / 2), MS.translate(10, 0));
MS.reveal("the origin ends up at", myGuess, MS.transformPoint(m, 0, 0).map(v => Math.round(v * 1e6) / 1e6));
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: a white line 1 pixel wide and fully opaque (alpha 255), blurred with σ = 2.
// What's the highest alpha left anywhere in the blurred line?
const myGuess = null; // TODO: your guess, 0 to 255
const line = MS.createFrame(21, 21);
MS.fillRect(line, 10, 0, 1, 21, [255, 255, 255]);
const blurred = MS.blur(line, 2);
let highest = 0;
for (let i = 3; i < blurred.data.length; i += 4) highest = Math.max(highest, blurred.data[i]);
MS.reveal("the highest alpha", myGuess, highest);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: a white square is drawn half a pixel to the right of the pixel grid, onto
// a transparent frame. Its left edge only half covers a column of pixels. What's the
// RED value of a pixel in that half-covered column?
const myGuess = null; // TODO: your guess, 0 to 255
const square = MS.createFrame(4, 4);
MS.clearFrame(square, [255, 255, 255]);
const out = MS.createFrame(8, 6);
MS.drawTransformed(out, square, MS.translate(2.5, 1));
MS.reveal("the red value at the edge", myGuess, MS.getPixel(out, 2, 2)[0]);
}


<details><summary>Answers</summary>

1. [0, 10]. The move happens first, so the origin goes to (10, 0); then the quarter turn swings it from "right" to "down". In the other order the origin would turn first (staying put, because it's the center of the turn) and then move to (10, 0).
2. 39, about 15% of what it was. A blur doesn't remove any ink, it spreads it: the same total alpha now covers a band about 13 pixels wide, so each pixel gets much less. That's why KineticIntro's blurred words look faint as well as soft.
3. 255. The pixel is half transparent (alpha 128), but still pure white. Averaged straight, the transparent pixel outside the square would have pulled it to 128, a grey fringe.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Multiply the other way round



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: what does the order inside multiply decide?
// TODO(break-it), in src/07-transforms.js → multiply:
//   swap m and n, so it does m first, then n.
const grow = (cx, cy, s) => MS.multiply(MS.translate(cx, cy), MS.multiply(MS.scale(s), MS.translate(-cx, -cy)));
for (const s of [1, 1.5, 2]) console.log("scale " + s + " around (50, 20): the middle goes to " + MS.transformPoint(grow(50, 20, s), 50, 20).join(", "));
// TODO: before the change, the middle stays put. After it, where does it go, and why does it move more as s grows?
}


#### 2. Paint forwards



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: why draw backwards?
// TODO(break-it), in src/07-transforms.js → drawTransformed:
//   instead, loop over src's pixels, move each pixel's center with m, and set the
//   dest pixel it lands on to that src pixel's color.
const dot = MS.createFrame(4, 4);
MS.clearFrame(dot, [255, 255, 255]);
const out = MS.createFrame(20, 20);
MS.drawTransformed(out, dot, MS.scale(4));
let painted = 0;
for (let y = 0; y < 20; y++) for (let x = 0; x < 20; x++) if (MS.getPixel(out, x, y)[3] > 0) painted++;
console.log("pixels painted: " + painted);
console.log(MS.asciiFrame(out));
// TODO: a 4 × 4 square scaled by 4 should cover 16 × 16 pixels. How many get painted after the change?
}


#### 3. Forget the alpha



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: what is premultiplying for?
// TODO(break-it), in src/07-transforms.js → boxBlur:
//   average each channel on its own, ignoring alpha (red is the average of the reds).
const sign = MS.createFrame(30, 15);
MS.fillRect(sign, 8, 4, 14, 7, [255, 220, 0]);
const soft = MS.blur(sign, 2);
console.log("a pixel just outside the yellow box: " + MS.getPixel(soft, 6, 7).join(", "));
// TODO: before the change it's yellow and faint. After it, what color is it? Picture it over a white page.
}


### Checkpoint

When all 18 tests pass, the checkpoint unlocks (layer 00 explains how it works). The picture is a word made of dots, leaning in italics through your own `transformPoint`. Every dot asks one small question about a rule, and a dot that gets a wrong answer flies off its letter. Its traps: matrices that use all six numbers, rotations past a quarter turn, `multiply` in both orders, matrices that can't be undone, pictures turned upside down and scaled up, sprites that only half cover a pixel, opacity over a background, blur radii that only just round up or down, and the edges of the frame. The three questions are below.



In [ ]:
// Checkpoint 07 · Transforms & blur. Replace each null with your answer.
MS.answers("07", {
  // 1. KineticIntro's frame is 1080 × 1920. Its words push in to scale 1.04 around the
  //    middle of the frame, (540, 960). Where does the frame's left edge (x = 0) end up?
  //    One decimal.
  q1: null,
  // 2. KineticIntro blurs its words in from blur(14px). Built from three box blurs, how
  //    many pixels wide is the area one pixel spreads over? Count every pixel that gets
  //    any of it, before anything is rounded to whole bytes.
  q2: null,
  // 3. m = multiply(translate(100, 50), multiply(rotate(Math.PI / 2), scale(2))).
  //    Which point does m move to (60, 70)? Answer as [x, y].
  q3: null,
});

In [ ]:
await checkpoint("07")


### In the real world

- Canvas keeps one of these matrices as its "current transform": [`translate()`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/translate), [`rotate()`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/rotate) and [`scale()`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/scale) each multiply it by a new matrix on the right, which is why the last one you call happens to your shapes first. [`DOMMatrix`](https://developer.mozilla.org/en-US/docs/Web/API/DOMMatrix) has the same six numbers as its `a` to `f`.
- CSS's [`transform-origin`](https://developer.mozilla.org/en-US/docs/Web/CSS/transform-origin) is the "move to the origin, scale, move back" trick, done for you. Its default is the element's center, which is why KineticIntro's `scale()` grows words from their middle.
- [`imageSmoothingEnabled`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/imageSmoothingEnabled) switches canvas between smooth sampling and nearest-neighbor, and CSS's [`image-rendering: pixelated`](https://developer.mozilla.org/en-US/docs/Web/CSS/image-rendering) does the same for images, which is how this course's pages show your frames with crisp pixels.
- Browsers hand `filter: blur()` to their graphics library (Chrome's is [Skia](https://skia.org/)), which picks its own fast way to approximate a Gaussian, often on the GPU.

### Going further (optional)

- **Read the original.** The Filter Effects spec's [`feGaussianBlur`](https://www.w3.org/TR/filter-effects-1/#feGaussianBlurElement) section gives its own recipe for three box blurs, with a box size chosen a different way, and a special case for even sizes. Work out its box size for σ = 14 and compare the spread with yours.
- Add `skew(ax, ay)` (CSS has [`skew()`](https://developer.mozilla.org/en-US/docs/Web/CSS/transform-function/skew)) and draw a word leaning by 15°.
- Make `drawTransformed` faster: when m is only a translation by whole pixels, no sampling is needed at all. How much faster is the third test picture?


## Layer 08 · Scenes

_A scene is a stack of pictures, each with its own clock._

**You'll write:** `drawTextCanvas`, `textSprite`, `sequence`, `series`, `camera`, `renderLayers`, `mulberry32`, `random` in `src/08-scenes.js`
**Builds on:** layers 00–07

KineticIntro is four scenes on one timeline: three words, then the logo. Each word is real text in Inter, the heaviest weight, turned into a picture and then moved, scaled, blurred and faded on its own schedule. This layer builds the parts for that: text from the browser's real fonts, clocks that start and stop, a camera, a stack of layers with effects, and randomness that gives the same answer every time you render.

### Under the hood

#### Real text is a picture the browser draws for you

Layer 02's bitmap font had one size and no curves. Real fonts are outlines, and the browser's text engine turns an outline into pixels, with smooth edges, at any size. You ask for it through a canvas: set [`ctx.font`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/font), then call [`fillText`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/fillText). The font is a string in the same order as CSS's [`font` shorthand](https://www.w3.org/TR/css-fonts-4/#font-prop): weight, then size, then family, like `"900 150px Inter"`.

Two things fail silently. A font string the canvas can't parse is simply ignored, as [the HTML standard](https://html.spec.whatwg.org/multipage/canvas.html#dom-context-2d-font) requires, so `"150px 900 Inter"` leaves you drawing in whatever font was there before, often `10px sans-serif`. And fonts load in the background: until Inter has arrived, the canvas quietly draws with a fallback font. The course pages ask the browser to load Inter when they open, and draw their pictures again once it's in.

To make a picture that fits the text, you measure first. [`measureText`](https://developer.mozilla.org/en-US/docs/Web/API/TextMetrics) returns the text's advance width and the font's own ascent and descent:

```text
   ┌─────────────────────────────┐  ┐
   │                             │  │
   │   D E S I G N .             │  │ fontBoundingBoxAscent
   ├─────────────────────────────┤  ┘ ◄── the baseline: letters sit on it
   │                             │  ┐ fontBoundingBoxDescent (room for g, p, y)
   └─────────────────────────────┘  ┘
   ◄────────── width ───────────►
```

The font's ascent and descent are the same for every word in that font and size, so "DESIGN." and "ANIMATE." get boxes of the same height with their baselines at the same place, and they line up. The pixels come back from [`getImageData`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/getImageData) as straight (not premultiplied) RGBA bytes, row by row: exactly the layout your frames have used since layer 00.

> **Study buddy:** "Ask me questions until I can explain what a font's ascent, descent and baseline are, and why sprites for two different words should use the font's ascent rather than each word's own ink. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2008%20%28Scenes%29%3A%20real%20fonts%2C%20sequences%2C%20a%20camera%2C%20masks%2C%20seeded%20randomness.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20what%20a%20font%27s%20ascent%2C%20descent%20and%20baseline%20are%2C%20and%20why%20sprites%20for%20two%20different%20words%20should%20use%20the%20font%27s%20ascent%20rather%20than%20each%20word%27s%20own%20ink.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2008%20%28Scenes%29%3A%20real%20fonts%2C%20sequences%2C%20a%20camera%2C%20masks%2C%20seeded%20randomness.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20what%20a%20font%27s%20ascent%2C%20descent%20and%20baseline%20are%2C%20and%20why%20sprites%20for%20two%20different%20words%20should%20use%20the%20font%27s%20ascent%20rather%20than%20each%20word%27s%20own%20ink.%20No%20code.)

#### Every layer has its own clock

A scene doesn't care what time the video says it is. It cares how long it has been running. Remotion's [`<Sequence>`](https://www.remotion.dev/docs/sequence) gives each part of a video its own frame counter: inside `<Sequence from={30} durationInFrames={20}>`, video frame 30 is frame 0, and outside frames 30–49 the sequence isn't drawn at all.

```text
  video frame:   0 ··· 29  30  31  ···  49  50 ···
  sequence:      (not on)   0   1  ···  19  (not on)
```

[`<Series>`](https://www.remotion.dev/docs/series) works out the `from`s for you by putting sequences back to back. An **offset** moves a sequence earlier (negative) or later (positive), and since each sequence starts where the one before it ends, everything after it moves too.

#### A camera is the world moving the other way

Pan a camera to the right and everything on screen slides left. Turn it clockwise and the world turns counterclockwise. A camera is the inverse of where it stands: to film from a camera at `(x, y)`, move the world by `(−x, −y)`, so the point the camera looks at is at the origin, turn it the opposite way to the camera, zoom, and then move the origin to the middle of the screen. Every layer's own matrix gets multiplied by the camera's, so one camera move moves the whole scene.

#### Layers, masks, and the order of effects

A scene is drawn like a painting: background first, each layer on top of the ones before. That's the [painter's algorithm](https://en.wikipedia.org/wiki/Painter%27s_algorithm). Each layer is a sprite plus instructions: where it goes, how see-through it is, how blurred, and which part of the screen it may touch.

```text
  sprite ──► blur ──► matrix ──► opacity ──► clip ──► onto the frame
          (in the sprite's       (fade)     (only inside
           own pixels)                        a rectangle)
```

Order matters, and CSS has an answer. The [Filter Effects spec](https://www.w3.org/TR/filter-effects-1/#FilterProperty) applies an element's filters "in the element's local coordinate system", before its transform moves it to the screen. So a word with `filter: blur(14px)` and `transform: scale(1.5)` gets a blur 1.5 times as wide. Blurring the sprite first, then transforming it, gives the same result.

A **clip** is the simplest mask: a rectangle the layer can't paint outside. CSS's `overflow: hidden` is one, and KineticIntro uses it to make each word rise out of an invisible slot.

#### Randomness you can replay

`Math.random()` gives a new number every time it's called. A video can't use it: Remotion renders frames in parallel, in several browser tabs at once, and [the docs warn](https://www.remotion.dev/docs/using-randomness) that each tab would get different random numbers, so a star could jump from frame to frame. frame = f(n) needs randomness that's a function too.

Remotion's [`random(seed)`](https://www.remotion.dev/docs/random) is one: the same seed always gives the same number. It turns a string into a 32-bit number with a hash, then scrambles that number with [mulberry32](https://gist.github.com/tommyettinger/46a874533244883189143505d203312c), a tiny generator by Tommy Ettinger. Both work on 32-bit integers, and JavaScript doesn't really have those. Its numbers are 64-bit floats, which hold whole numbers exactly only up to 2⁵³. The bit operators (`^`, `|`, `>>>`) first convert to 32-bit integers, and [`Math.imul`](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Math/imul) multiplies two 32-bit integers the way a CPU does, keeping the low 32 bits of the answer. Plain `*` works out the full product first, and once that passes 2⁵³ the low bits are already gone.

```text
  0x7FFFFFFF × 0x7FFFFFFF = 4611686014132420609   (needs 62 bits)
  as a float:               4611686014132420608   (the low bits are lost)
```

### Your turn

Work through `src/08-scenes.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **drawTextCanvas.** In what order do weight, size and family go? How do you turn an alpha of 0–255 into the 0–1 that `rgba()` wants?
2. **textSprite.** Which font must be set before you call `measureText`? Where's the baseline in the sprite, if there's `pad` empty pixels above the font's ascent? When `MS.createCanvas` gives you `null`, which layer 02 function tells you the width?
3. **sequence.** For a sequence from frame 30 lasting 20 frames, what's its last frame? Is frame 50 inside?
4. **series.** Where does the second sequence start? If the second one has an offset, where does the third start?
5. **camera.** Where should the point the camera looks at end up? If the camera turns clockwise, which way should the world turn? You have `translate`, `scale`, `rotate` and `multiply` from layer 07: in which order does a world point need them?
6. **renderLayers.** When should the blur happen, before or after the matrix? How can you stop a layer from painting outside its clip, using only what you already have?
7. **mulberry32 and random.** Which steps need `Math.imul`, and what does `>>> 0` do to a negative number?

Run the tests after each function. A failing test tells you exactly what it expected.

Once it works, the test pictures show text sprites with their boxes and baselines (real fonts in a browser, your bitmap font elsewhere), 300 dots scattered by seeded randomness, and three words playing in a series under a drifting camera.

> **Study buddy:** "Give me a timeline with three sequences and some offsets, and ask me which sequences are on at a few frames, and what each one's own frame number is. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2008%20%28Scenes%29%3A%20real%20fonts%2C%20sequences%2C%20a%20camera%2C%20masks%2C%20seeded%20randomness.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20a%20timeline%20with%20three%20sequences%20and%20some%20offsets%2C%20and%20ask%20me%20which%20sequences%20are%20on%20at%20a%20few%20frames%2C%20and%20what%20each%20one%27s%20own%20frame%20number%20is.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2008%20%28Scenes%29%3A%20real%20fonts%2C%20sequences%2C%20a%20camera%2C%20masks%2C%20seeded%20randomness.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20a%20timeline%20with%20three%20sequences%20and%20some%20offsets%2C%20and%20ask%20me%20which%20sequences%20are%20on%20at%20a%20few%20frames%2C%20and%20what%20each%20one%27s%20own%20frame%20number%20is.%20No%20code.)



### ✏️ Your code for layer 08

This cell is your `src/08-scenes.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2008%20%28Scenes%29%3A%20real%20fonts%2C%20sequences%2C%20a%20camera%2C%20masks%2C%20seeded%20randomness.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2008%20%28Scenes%29%3A%20real%20fonts%2C%20sequences%2C%20a%20camera%2C%20masks%2C%20seeded%20randomness.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 08 · Scenes — real fonts, sequences, a camera, masks, seeded randomness.
// Read the chapter first (docs/08-scenes.md). Each comment below is a spec; the tests check it.
(function (MS) {
  // Text options, used by both text functions:
  //   { size, weight = 400, family = "sans-serif", color = [255, 255, 255, 255], letterSpacing = 0, pad = 0 }
  // size and letterSpacing are in pixels; color is [r, g, b] or [r, g, b, a] with 0–255 values.

  /**
   * Draw `text` on a 2D canvas context, with its baseline's left end at (x, y).
   * Before calling ctx.fillText(text, x, y), set:
   *   ctx.font          weight, size and family as CSS writes them: "900 40px Inter"
   *   ctx.letterSpacing the spacing in px, like "-2px" (and "0px" for none)
   *   ctx.fillStyle     the color as "rgba(r, g, b, alpha)", alpha from 0 to 1:
   *                     [91, 140, 255, 51] → "rgba(91, 140, 255, 0.2)"
   *   ctx.textBaseline  "alphabetic", and ctx.textAlign "left"
   */
  function drawTextCanvas(ctx, text, x, y, opts) {
    throw MS.todo("drawTextCanvas");
  }

  /**
   * A frame with `text` in it, sized to fit, plus a `baseline` property: the
   * y of the text's baseline inside the frame.
   * With a canvas (MS.createCanvas(w, h) returns one in browsers, null elsewhere):
   * - measure the text with ctx.measureText, using the same font and spacing
   *   you'll draw with;
   * - width = Math.ceil(metrics.width) + 2 × pad,
   *   height = Math.ceil(metrics.fontBoundingBoxAscent + metrics.fontBoundingBoxDescent) + 2 × pad;
   * - draw it with drawTextCanvas on a canvas that size, at x = pad and
   *   y = pad + fontBoundingBoxAscent (that's also the baseline);
   * - copy the canvas's pixels (ctx.getImageData) into a new frame.
   * With no canvas, use your bitmap font from layer 02: scale = Math.round(size ÷ 10)
   * (at least 1), the frame is the text's width at that scale plus 2 × pad by
   * 7 × scale + 2 × pad, the text is drawn at (pad, pad), and the baseline is
   * pad + 7 × scale. The bitmap font ignores weight, family and letterSpacing.
   */
  function textSprite(text, opts) {
    throw MS.todo("textSprite");
  }

  /**
   * Like Remotion's <Sequence from durationInFrames>: for frame n of the video,
   * the sequence's own frame number, counting from 0 at frame `from`. Return
   * null when n is outside the sequence: before `from`, or at or after
   * from + durationInFrames. With no duration, it never ends.
   */
  function sequence(n, from, durationInFrames) {
    throw MS.todo("sequence");
  }

  /**
   * Like Remotion's <Series>: items is a list of { durationInFrames, offset }.
   * Each item starts when the one before it ends (the first at frame 0), moved
   * by its offset (default 0; negative overlaps the one before). An offset moves
   * every item after it too. Return a new list of { from, durationInFrames }.
   */
  function series(items) {
    throw MS.todo("series");
  }

  /**
   * The matrix that films the world with a camera, for a screen width × height.
   * The camera looks at world point (x, y), which lands in the middle of the
   * screen. zoom scales everything around the middle. rotation (radians) turns
   * the camera clockwise, so the world on screen turns the other way.
   * Defaults: x 0, y 0, zoom 1, rotation 0.
   */
  function camera({ x, y, zoom, rotation }, width, height) {
    throw MS.todo("camera");
  }

  /**
   * Draw layers onto `frame` in order (later ones on top) and return `frame`.
   * Each layer is { sprite, matrix, opacity, blur, clip }:
   *   sprite   a frame
   *   matrix   where the sprite goes, as for drawTransformed (default: no change)
   *   opacity  0 to 1 (default 1); a layer with opacity 0 draws nothing
   *   blur     a σ for your blur (default 0: none), applied to the sprite
   *            BEFORE the matrix moves it
   *   clip     optional { x, y, width, height } in frame pixels (whole numbers):
   *            the layer only changes pixels inside this rectangle
   */
  function renderLayers(frame, layers) {
    throw MS.todo("renderLayers");
  }

  /**
   * mulberry32, one step, exactly as Remotion does it. All of it is 32-bit
   * integer math: "×" is Math.imul (a 32-bit multiply), ^ | >>> are JavaScript's
   * bit operators.
   *   t = a + 0x6D2B79F5
   *   t = (t ^ (t >>> 15)) × (t | 1)
   *   t = t ^ (t + (t ^ (t >>> 7)) × (t | 61))
   *   return ((t ^ (t >>> 14)) >>> 0) ÷ 4294967296      (a number from 0 up to 1)
   */
  function mulberry32(a) {
    throw MS.todo("mulberry32");
  }

  /**
   * Remotion's random(seed): the same seed always gives the same number from
   * 0 up to 1.
   * - a string: hash it (below), then return mulberry32(hash);
   * - a number: return mulberry32(seed × 10000000000);
   * - null: return Math.random() (really random: avoid it in a video);
   * - anything else: throw an Error.
   * The hash (Java's String.hashCode): start at 0; for each character,
   *   hash = (hash × 31 + its char code), kept to a 32-bit integer with | 0.
   */
  function random(seed) {
    throw MS.todo("random");
  }

  Object.assign(MS, { drawTextCanvas, textSprite, sequence, series, camera, renderLayers, mulberry32, random });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("08")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: a sequence starts at frame 60 and lasts 30 frames.
// Is it still on at video frame 90?
const myGuess = null; // TODO: true or false
MS.reveal("on at frame 90", myGuess, MS.sequence(90, 60, 30) !== null);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: do a plain multiply and Math.imul agree on this one?
const myGuess = null; // TODO: true or false
const a = 0x6D2B79F5;
MS.reveal("(a * a) | 0 equals Math.imul(a, a)", myGuess, ((a * a) | 0) === Math.imul(a, a));
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: a 400 × 300 screen. The camera looks at (0, 0) and is turned half a turn.
// Where on the screen does the world point (30, 10) appear?
const myGuess = null; // TODO: your guess, as [x, y]
const p = MS.transformPoint(MS.camera({ rotation: Math.PI }, 400, 300), 30, 10);
MS.reveal("(30, 10) appears at", myGuess, p.map(v => Math.round(v * 1e6) / 1e6));
}


<details><summary>Answers</summary>

1. false. The sequence covers frames 60 to 89: 30 frames. Frame 90 is the first frame after it, the same way an array of length 30 has no index 30, so `sequence(90, 60, 30)` is null.
2. false. a × a is about 3.4 × 10¹⁸, far past 2⁵³, so the float has already lost its low bits before `| 0` keeps only the low 32. `Math.imul` keeps them exactly. That's the whole reason mulberry32 uses it.
3. [170, 140]. Half a turn sends (30, 10) to (−30, −10), so the point appears 30 to the left of and 10 above the middle (200, 150). A half turn looks the same whichever way the camera turns; try a quarter turn to tell the two apart.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Include the last frame



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: why is the end of a sequence not included?
// TODO(break-it), in src/08-scenes.js → sequence:
//   count frame from + durationInFrames as inside, too.
const parts = MS.series([{ durationInFrames: 30 }, { durationInFrames: 30 }, { durationInFrames: 30 }]);
for (const n of [29, 30, 31, 59, 60]) {
  const on = parts.map((p, i) => MS.sequence(n, p.from, p.durationInFrames) === null ? null : "word " + (i + 1)).filter(Boolean);
  console.log("frame " + n + ": " + on.join(" and "));
}
// TODO: before the change, exactly one word is on at every frame. After it, what happens at frames 30 and 60?
}


#### 2. Multiply with *



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: does Math.imul matter?
// TODO(break-it), in src/08-scenes.js → mulberry32:
//   use * instead of Math.imul for both multiplications.
console.log("random(\"hello\") = " + MS.random("hello") + "   (Remotion says 0.41293257870711386)");
let low = 0;
for (let i = 0; i < 1000; i++) if (MS.random("s" + i) < 0.5) low++;
console.log("below 0.5: " + low + " of 1000");
// TODO: which line changes? Are the numbers still spread evenly? Would your video still match Remotion's?
}


#### 3. Blur after moving



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: why blur before the matrix?
// TODO(break-it), in src/08-scenes.js → renderLayers:
//   move the sprite first (into a frame the size of the output), then blur that.
const word = MS.textSprite("HI", { size: 20, pad: 6 });
const frame = MS.createFrame(70, 40);
const t0 = Date.now();
MS.renderLayers(frame, [{ sprite: word, matrix: MS.multiply(MS.translate(10, 4), MS.scale(2)), blur: 2 }]);
console.log("took " + (Date.now() - t0) + " ms");
console.log(MS.asciiFrame(frame));
// TODO: is the blur as soft as before? How did the time change, and why (think of how many pixels get blurred)?
}


### Checkpoint

When all 22 tests pass, the checkpoint unlocks (layer 00 explains how it works). The picture is a word made of dots, filmed through your own camera. Every dot asks one small question about a rule, and a dot that gets a wrong answer flies off its letter. Text is checked on a pretend canvas, so the picture is the same on every computer. Its traps: sequences that start before frame 0, last one frame, or never end; offsets on the first sequence of a series; cameras turned both ways and zoomed out; seeds with accents in them and huge numbers for mulberry32; layers that are invisible, clipped or blurred; and text that's sized, spaced and measured exactly. The three questions are below.



In [ ]:
// Checkpoint 08 · Scenes. Replace each null with your answer.
MS.answers("08", {
  // 1. KineticIntro's three words each last 48 frames, and each starts 40 frames after
  //    the one before. As a series, what offset do the second and third words need?
  q1: null,
  // 2. What is random(0)? Three decimals.
  q2: null,
  // 3. A 1080 × 1920 screen. The camera looks at (100, 0) and is turned a quarter turn
  //    clockwise (rotation Math.PI / 2), zoom 1. Where does the world point (0, 0)
  //    appear? Answer as [x, y].
  q3: null,
});

In [ ]:
await checkpoint("08")


### In the real world

- Remotion loads web fonts with [`@remotion/google-fonts`](https://www.remotion.dev/docs/google-fonts/load-font), and its [`delayRender()`](https://www.remotion.dev/docs/delay-render) makes a render wait until something asynchronous, like a font, is ready. Without it, the first frames could be drawn in a fallback font.
- [`useCurrentFrame()`](https://www.remotion.dev/docs/use-current-frame) inside a `<Sequence>` returns the sequence's own frame, exactly like `sequence()`. Remotion also has [`<Freeze>`](https://www.remotion.dev/docs/freeze) and [`<Loop>`](https://www.remotion.dev/docs/loop), which change the frame number in other ways.
- 2D engines like [PixiJS](https://pixijs.com/) and game engines keep a scene graph: every object has a matrix relative to its parent, and the camera is one more matrix at the top.
- CSS has many masks: `overflow: hidden` and [`clip-path`](https://developer.mozilla.org/en-US/docs/Web/CSS/clip-path) cut shapes, and [`mask-image`](https://developer.mozilla.org/en-US/docs/Web/CSS/mask-image) uses a picture's alpha.

### Going further (optional)

- **Read the original.** The HTML standard's [text preparation algorithm](https://html.spec.whatwg.org/multipage/canvas.html#text-preparation-algorithm) is what `fillText` and `measureText` run. Look for where `letterSpacing` and `textBaseline` come in.
- Add `rotate` and a center point to `camera`'s options, and make a camera that orbits a word.
- Write `noise(x)`: smooth randomness that changes gently as x changes, by blending `random()` values at whole numbers with your easing from layer 05. Use it to make the stars twinkle.


## Layer 09 · KineticIntro

_Everything you've built, on one timeline._

**You'll write:** `sceneState`, `maskUp`, `linearGradientT`, `blendOverlay`, `background`, `vignette`, `wordLayer`, `lockupLayer`, `kineticIntro` in `src/09-kinetic-intro.js`
**Builds on:** layers 00–08

This is the layer the whole course has been heading for. KineticIntro, from [Claude Motion Studio](https://github.com/MoussaabBadla/claude-motion-studio), is six seconds of portrait video: **DESIGN.**, **ANIMATE.** and **RENDER.** rise out of nowhere, push toward you and blur away, then **MOTION·STUDIO** settles in with a sweep of light and a subtitle. In Remotion it's a few React components styled with CSS, and Chrome draws it. Here you turn that CSS into calls of your own functions: every gradient, blur, curve and blend is something you've built. The numbers come from KineticIntro's source; where CSS leaves a choice to the browser (exactly where flexbox puts a box, how text is antialiased), this layer says what it chose.

### Under the hood

#### Reading a component as math

KineticIntro wraps each word in a `Scene` component. Stripped of React, a Scene is a handful of `interpolate` calls on the frame number, and three numbers that come out of them: how big, how see-through and how blurred.

```text
  word 2, "ANIMATE.": a = 40, b = 88
  frame:     38  40                        66                      88  90
  enter:          0 ─── APPLE, 26 frames ──► 1
  exit:                                    0 ─── SMOOTH, 22 frames ──► 1
  push:           1 ──────────────────────────────────────────────────► 1.04
  visible:   ◄─────────────────────────────────────────────────────────────►
```

`scale` multiplies three things: growing from `enterFrom` while it enters, the slow push, and growing toward `exitTo` while it leaves. `opacity` is the smaller of "how far in" and "how far from out". The blur is the leftover of the entrance plus the start of the exit. A blur below 0.15 px is invisible, so KineticIntro turns it off, and that saves Chrome a filter pass on most frames.

Notice what isn't there: no state, no "previous frame". Each number is a function of n. That's layer 03's frame = f(n), all the way up.

#### A mask is a window

Inside each Scene, a `MaskUp` holds the word in a box with `overflow: hidden`, and slides the word up from 110% of its own height below the box to its place. The box is a window: you only see the part of the word that's inside it.

```text
   frame 42 (start)         frame 50               frame 64 (home)
   ┌────────────────┐       ┌────────────────┐      ┌────────────────┐
   │                │       │ ▄▄  ▄  ▄ ▄ ▄▄▄ │      │ █▀█ █▄ █ █ █▀▄ │
   └────────────────┘       └────────────────┘      └────────────────┘
     ANIMATE.  (hidden,       ▀▀  ▀  ▀ ▀ ▀▀▀
     below the window)        (the rest still hidden)
```

The order of effects matters again. The window is inside the Scene, and the Scene's blur applies to everything in it, so the word is cut by the window first and blurred afterwards: the cut edge goes soft. That's why the window belongs in the sprite, before `renderLayers` blurs it. The sprite needs empty padding around the window, too, or the blur would have nowhere to spread.

#### Where CSS puts a gradient

The light sweep is a `linear-gradient(100deg, …)`. [CSS Images](https://www.w3.org/TR/css-images-3/#linear-gradients) defines exactly where such a gradient runs. `0deg` points up and angles turn clockwise, so `90deg` points right and `100deg` points right and a little down. The **gradient line** goes through the middle of the box in that direction, and its length is chosen so that the corners land exactly on 0% and 100%:

```text
     0% ●─────────────────────────┐
        │ ╲                        │      the gradient line points along the angle
        │   ╲                      │      (here 100deg: right, a little down) and
        │     ●  the box's middle  │      passes through the middle; lines of
        │       ╲                  │      equal color cross it at right angles
        └─────────╲────────────────● 100%
                   ▼
      length = |W·sin A| + |H·cos A|: just enough to reach both corners
```

A point's position along the gradient is how far along that line its shadow falls. The background's `radial-gradient(70% 60% at 50% 38%, …)` is simpler: an ellipse with radii of 70% of the width and 60% of the height, centered half way across and 38% down, which is exactly what your `fillRadialGradient` takes. A stop like `transparent 60%` ends the gradient early: past 60% of the way out, it's transparent. CSS mixes gradient colors premultiplied, so fading to `transparent` (which is black at alpha 0) never darkens anything, as you saw in layer 07.

> **Study buddy:** "Give me a box, a gradient angle and a point, and ask me to sketch where the gradient line goes and roughly what percentage the point is at. Then ask me why the line is longer at 45deg than at 90deg. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2009%20%28KineticIntro%29%3A%20the%20recreation.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20a%20box%2C%20a%20gradient%20angle%20and%20a%20point%2C%20and%20ask%20me%20to%20sketch%20where%20the%20gradient%20line%20goes%20and%20roughly%20what%20percentage%20the%20point%20is%20at.%20Then%20ask%20me%20why%20the%20line%20is%20longer%20at%2045deg%20than%20at%2090deg.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2009%20%28KineticIntro%29%3A%20the%20recreation.%20Here%27s%20what%20I%20want%20help%20with%3A%20Give%20me%20a%20box%2C%20a%20gradient%20angle%20and%20a%20point%2C%20and%20ask%20me%20to%20sketch%20where%20the%20gradient%20line%20goes%20and%20roughly%20what%20percentage%20the%20point%20is%20at.%20Then%20ask%20me%20why%20the%20line%20is%20longer%20at%2045deg%20than%20at%2090deg.%20No%20code.)

#### Blend modes: changing what "over" means

Layer 01's blend put the new color over the old one. CSS's [`mix-blend-mode`](https://www.w3.org/TR/compositing-1/#mix-blend-mode) keeps that recipe but changes one ingredient: where the two colors overlap, the result is `B(backdrop, source)` instead of just the source. The [general formula](https://www.w3.org/TR/compositing-1/#generalformula) mixes three parts, by how much each color covers:

```text
  only the source shows:     (1 − αb) × αs × Cs
  both overlap:              αb × αs × B(Cb, Cs)       ← the blend mode lives here
  only the backdrop shows:   (1 − αs) × αb × Cb
```

With `B(Cb, Cs) = Cs` that's exactly your `blendPixel`. KineticIntro's sweep uses [overlay](https://www.w3.org/TR/compositing-1/#blendingoverlay): where the backdrop is dark it multiplies, which makes dark darker; where it's light it "screens", which makes light lighter. So a white sweep brightens the light letters a lot and the dark background hardly at all.

Which backdrop, though? A blend mode only blends with the content of its own **stacking context**, and the Scene's `transform` makes it one. So the sweep never touches the page's background: it blends with the lockup's own pixels, and where those are transparent, it's an ordinary half-white blend. That's why `lockupLayer` blends the sweep into its sprite, not into the frame.

#### A shadow is a blur

The vignette is `box-shadow: inset 0 0 280px rgba(0, 0, 0, 0.7)`. An inset shadow is the shadow of everything *outside* the box, falling inward. [CSS Backgrounds](https://www.w3.org/TR/css-backgrounds-3/#shadow-blur) says a blur radius of 280px means a Gaussian blur with σ = 140, half the radius. So: make a frame that's black everywhere except a hole the size of the screen, blur it with your three box blurs, and keep the middle. The margin around the hole must be wider than the blur reaches, or the far edge of the black would fade in too. Three boxes of radius r reach 3r pixels, which is why the spec asks for a little more than 3σ.

#### Make it once

At full size, the vignette blurs a frame of more than five million pixels. If `render` did that for every frame, KineticIntro would take minutes. But frame = f(n) only says a frame must depend on n *alone*; anything that doesn't depend on n at all can be made once and reused. The background and the vignette never change, so `kineticIntro` makes them when the composition is made, and each frame starts from a copy.

### Your turn

Work through `src/09-kinetic-intro.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **sceneState.** Which of your layer 04 options gives "clamp" at both ends, and which takes an easing? At frame `a`, what should each of the four numbers be?
2. **maskUp.** What's the value one frame before `at`? One frame after `at + dur`?
3. **linearGradientT.** Which unit vector points in the direction of the angle, with 0deg up and y pointing down? How far along it is the point, measured from the middle of the box?
4. **blendOverlay.** Which color decides between "multiply" and "screen": the source or the backdrop? What happens to the formula when the backdrop is fully transparent?
5. **background.** What radii and center does each gradient give `fillRadialGradient`? How do you draw the glow over the background without wiping it out?
6. **vignette.** How do you make a frame that's opaque black with a transparent hole? After blurring, which matrix cuts the middle out?
7. **wordLayer.** Where does the word go inside its window at frame `a + 2`? Why does the sprite need padding, and why is that padding's size based on blur 16?
8. **lockupLayer.** In what order do the title, the sweep and the subtitle go into the sprite? Where's the sweep's box?
9. **kineticIntro.** What can you make once, outside `render`?

Run the tests after each function. A failing test tells you exactly what it expected.

Once it works, the test pictures show KineticIntro itself, playing, and three of its pieces on their own.

> **Study buddy:** "Show me the CSS for one KineticIntro Scene and ask me to translate each property into one of my functions, one at a time. Only tell me which layer to look in if I'm stuck. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2009%20%28KineticIntro%29%3A%20the%20recreation.%20Here%27s%20what%20I%20want%20help%20with%3A%20Show%20me%20the%20CSS%20for%20one%20KineticIntro%20Scene%20and%20ask%20me%20to%20translate%20each%20property%20into%20one%20of%20my%20functions%2C%20one%20at%20a%20time.%20Only%20tell%20me%20which%20layer%20to%20look%20in%20if%20I%27m%20stuck.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2009%20%28KineticIntro%29%3A%20the%20recreation.%20Here%27s%20what%20I%20want%20help%20with%3A%20Show%20me%20the%20CSS%20for%20one%20KineticIntro%20Scene%20and%20ask%20me%20to%20translate%20each%20property%20into%20one%20of%20my%20functions%2C%20one%20at%20a%20time.%20Only%20tell%20me%20which%20layer%20to%20look%20in%20if%20I%27m%20stuck.%20No%20code.)



### ✏️ Your code for layer 09

This cell is your `src/09-kinetic-intro.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2009%20%28KineticIntro%29%3A%20the%20recreation.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2009%20%28KineticIntro%29%3A%20the%20recreation.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 09 · KineticIntro — the recreation.
// Read the chapter first (docs/09-kinetic-intro.md). Each comment below is a spec; the tests check it.
(function (MS) {
  // KineticIntro's numbers. Its frame is 1080 × 1920; at another width, multiply
  // every length in pixels (font sizes, spacing, blurs, gaps) by k = width ÷ 1080.
  //   colors   ink #EAF0FA, muted #8A93A6, primary #5B8CFF
  //   easing   APPLE = cubic-bezier(0.16, 1, 0.3, 1), SMOOTH = cubic-bezier(0.6, 0, 0.2, 1)
  //   "clamp"  means extrapolateLeft and extrapolateRight are both "clamp"
  // Tip: make your easing functions inside the functions that use them (or the
  // first time they're needed), not when this file loads.

  /**
   * Where a KineticIntro Scene is at frame n. The scene runs from frame a to b.
   * Options (defaults): a, b, enterFrom (1), exitTo (1), blurIn (0), blurOut (0),
   * enterDur (26), exitDur (22), pushTo (1.04).
   *   enter   = interpolate(n, [a, a + enterDur], [0, 1]), clamped, eased with APPLE
   *   exit    = interpolate(n, [b − exitDur, b], [0, 1]), clamped, eased with SMOOTH
   *   push    = interpolate(n, [a, b], [1, pushTo]), clamped
   * Return { visible, scale, opacity, blur }:
   *   visible = n is from a − 2 to b + 2 (both included)
   *   scale   = lerp(enterFrom, 1, enter) × push × lerp(1, exitTo, exit)
   *   opacity = the smaller of enter and 1 − exit
   *   blur    = (1 − enter) × blurIn + exit × blurOut, or 0 if that's 0.15 or less
   */
  function sceneState(n, opts) {
    throw MS.todo("sceneState");
  }

  /**
   * KineticIntro's MaskUp: how far below its slot a word is at frame n, as a
   * percentage of the word's own height. It rises from 110 at frame `at` to 0 at
   * frame at + dur (default 22), clamped, eased with APPLE.
   */
  function maskUp(n, at, dur) {
    throw MS.todo("maskUp");
  }

  /**
   * Where the point (x, y) of a w × h box falls along a CSS linear-gradient(deg),
   * as a number from 0 (where the gradient starts) to 1 (where it ends).
   * 0deg points up, 90deg points right (angles go clockwise). The gradient line
   * runs through the box's center in that direction, and is just long enough
   * that its two ends touch the box's corners: |w × sin| + |h × cos| long.
   */
  function linearGradientT(x, y, w, h, deg) {
    throw MS.todo("linearGradientT");
  }

  /**
   * Blend `color` onto pixel (x, y) like CSS's mix-blend-mode: overlay. Like your
   * blendPixel (layer 01), but where the two overlap, the color becomes
   * B(backdrop, color) instead of the color itself. With every channel as a
   * number from 0 to 1 (divide by 255), for each of red, green and blue:
   *   B = 2 × Cb × Cs                      if the backdrop Cb ≤ 0.5
   *   B = 1 − 2 × (1 − Cb) × (1 − Cs)      otherwise
   *   αo = αs + αb × (1 − αs)
   *   Co = ((1 − αb) × αs × Cs + αb × αs × B + (1 − αs) × αb × Cb) ÷ αo
   * (s is the color you blend, b the pixel already there, o the result.) Store
   * Co and αo back as 0–255. Pixels outside the frame, and a result with αo = 0,
   * leave the frame unchanged.
   */
  function blendOverlay(frame, x, y, color) {
    throw MS.todo("blendOverlay");
  }

  /**
   * KineticIntro's background, as a new width × height frame, using your
   * fillRadialGradient (layer 01):
   *   radial-gradient(70% 60% at 50% 38%, #1A2740, #0B0D12)
   *   then over it, blended normally,
   *   radial-gradient(60% 40% at 50% 45%, rgba(91, 140, 255, 0.22), transparent 60%)
   * "70% 60%" are the ellipse's x and y radii as fractions of width and height.
   * "transparent 60%" means the glow has faded out 60% of the way to its edge.
   * Use alpha 56 for 0.22, and fade the glow to the same blue with alpha 0.
   */
  function background(width, height) {
    throw MS.todo("background");
  }

  /**
   * An inset box-shadow, as a width × height frame: CSS's
   *   box-shadow: inset 0 0 <radius>px rgba(0, 0, 0, alpha)
   * A blur radius of r means a Gaussian blur with σ = r ÷ 2. Make a frame that's
   * the size of the screen plus a margin of Math.ceil(3σ) + 2 on every side,
   * opaque black except for a transparent hole the size of the screen in the
   * middle. Blur it with your blur(σ), then cut the middle out (drop the margin)
   * with its alpha multiplied by `alpha`.
   */
  function vignette(width, height, radius, alpha) {
    throw MS.todo("vignette");
  }

  /**
   * The layer (for renderLayers) of word i at frame n, in a frame `width` wide
   * (and Math.round(width × 16 ÷ 9) high); null when the word's scene isn't visible.
   * The words are "DESIGN.", "ANIMATE.", "RENDER.". Word i's scene: a = 40 × i,
   * b = a + 48, enterFrom 0.8, exitTo 1.5, blurIn 14, blurOut 16.
   * - The text: textSprite, 150 px, weight 900, family "Inter", ink, letterSpacing −4 px.
   * - The mask: a frame the size of the text sprite, with the text drawn into it
   *   moved down by maskUp(n, a + 2) percent of the text's height. Anything that
   *   lands outside it is cut off.
   * - The sprite: that mask frame in the middle of a bigger frame with
   *   pad = Math.ceil(3 × 16 × k) + 2 empty pixels on every side, room for the blur.
   * - The layer: { sprite, matrix, opacity, blur }, where the matrix scales the
   *   sprite by the scene's scale around the sprite's center and puts that center
   *   in the middle of the frame, opacity is the scene's, and blur is the scene's × k.
   * (Remember: renderLayers blurs before the matrix, like CSS.)
   */
  function wordLayer(n, i, width) {
    throw MS.todo("wordLayer");
  }

  /**
   * The lockup's layer at frame n (null when its scene isn't visible). Its scene:
   * a = 120, b = 180, enterFrom 0.86, exitTo 1, blurIn 14, exitDur 2.
   * - Title: textSprites "MOTION", "·", "STUDIO" side by side (96 px, weight 900,
   *   "Inter", letterSpacing −3 px); the "·" is primary, the rest ink.
   * - Subtitle: textSprite "built in code" (36 px, weight 500, "Inter", muted,
   *   letterSpacing 2 px), Math.round(22 × k) below the title, faded by
   *   interpolate(n, [150, 168], [0, 1]), clamped.
   * - The sprite: the title above the subtitle, each centered (round positions
   *   to whole pixels), with pad = Math.ceil(3 × 14 × k) + 2 empty pixels around.
   * - The light sweep, after the title and before the subtitle: over the title's
   *   box grown by Math.round(16 × k) on every side, for each pixel, with p its
   *   center's linearGradientT(…, 100deg) × 100 and s = interpolate(n, [140, 172],
   *   [−20, 130]) clamped, blendOverlay white with alpha 0.5 × max(0, 1 − |p − s| ÷ 12),
   *   in 0–255 (skip pixels where that's 0).
   * - The layer: as for the words, from the lockup's scene.
   */
  function lockupLayer(n, width) {
    throw MS.todo("lockupLayer");
  }

  /**
   * KineticIntro: a composition `width` wide (default 1080), Math.round(width × 16 ÷ 9)
   * high, 30 fps, 180 frames. Each frame is the background, then with your
   * renderLayers: the three words, the lockup, and on top the vignette
   * (radius 280 × k, alpha 0.7). The background and vignette never change: make
   * them once, when the composition is made, not for every frame.
   */
  function kineticIntro(width) {
    throw MS.todo("kineticIntro");
  }

  Object.assign(MS, { sceneState, maskUp, linearGradientT, blendOverlay, background, vignette, wordLayer, lockupLayer, kineticIntro });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("09")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: "DESIGN." runs from frame 0 to 48. Half way, at frame 24, how big is it?
const myGuess = null; // TODO: your guess, to 3 decimals (1 = its normal size)
const s = MS.sceneState(24, { a: 0, b: 48, enterFrom: 0.8, exitTo: 1.5, blurIn: 14, blurOut: 16 });
MS.reveal("the scale at frame 24", myGuess, Math.round(s.scale * 1000) / 1000);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: the vignette at half size (540 × 960, blur radius 140, alpha 0.7).
// What's the alpha of the very top-left pixel, from 0 to 255?
const myGuess = null; // TODO: your guess
MS.reveal("the corner's alpha", myGuess, MS.getPixel(MS.vignette(540, 960, 140, 0.7), 0, 0)[3]);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: overlay opaque black onto a light grey [200, 200, 200].
// What's the red value afterwards?
const myGuess = null; // TODO: your guess, 0 to 255
const f = MS.createFrame(1, 1);
MS.clearFrame(f, [200, 200, 200]);
MS.blendOverlay(f, 0, 0, [0, 0, 0]);
MS.reveal("red after overlaying black", myGuess, MS.getPixel(f, 0, 0)[0]);
}


<details><summary>Answers</summary>

1. 1.02. It finished entering a few frames ago (26 frames of APPLE get very close to 1 early), and the push has been growing it the whole time: 4% over 48 frames is 2% by half way. The push is so slow you never see it as motion, only as the word feeling alive.
2. 134, about 0.7 × ¾ of 255. A corner pixel is close to two edges, so the blurred black reaches it from two sides: only about a quarter of the area around it is the transparent hole. Halfway down an edge it's about half, so the alpha there is about 0.7 × ½ of 255.
3. 145. Over a light backdrop, overlay screens, so even black only pulls 200 down to 145: overlay keeps the light parts light. Over a dark grey, the same black would give pure black.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. A window with no frame



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: why does the window go in the sprite?
// TODO(break-it), in src/09-kinetic-intro.js → wordLayer:
//   skip the window: draw the text straight into the padded sprite, moved down by maskUp, and
//   leave out the frame that cuts it off.
const layer = MS.wordLayer(50, 1, 216);
let below = 0;
const s = layer.sprite, bottom = Math.floor(s.height * 0.75);
for (let y = bottom; y < s.height; y++) for (let x = 0; x < s.width; x++) below += s.data[(y * s.width + x) * 4 + 3];
console.log("ink in the bottom quarter of the sprite at frame 50: " + below);
// TODO: before the change, how much ink is below the window? After it, what would the viewer see under the word?
}


#### 2. Overlay, backwards



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: whose color decides?
// TODO(break-it), in src/09-kinetic-intro.js → blendOverlay:
//   choose between the two formulas with the SOURCE color (Cs ≤ 0.5) instead of the backdrop's.
for (const grey of [30, 100, 200]) {
  const f = MS.createFrame(1, 1);
  MS.clearFrame(f, [grey, grey, grey]);
  MS.blendOverlay(f, 0, 0, [255, 255, 255, 128]);
  console.log("half-white over grey " + grey + " → " + MS.getPixel(f, 0, 0)[0]);
}
// TODO: before the change, the white lifts light greys more than dark ones. After it? (That other blend mode has a name: hard-light.)
}


#### 3. Make it every frame



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: what does "make it once" save?
// TODO(break-it), in src/09-kinetic-intro.js → kineticIntro:
//   make the background and the vignette inside render, for every frame.
const comp = MS.kineticIntro(216);
const t0 = Date.now();
for (let n = 0; n < 180; n += 18) MS.renderFrame(comp, n);
console.log("10 frames: " + (Date.now() - t0) + " ms");
// TODO: how much slower is it after the change? How long would all 180 frames take at 1080 wide?
}


### Checkpoint

When all 21 tests pass, the checkpoint unlocks (layer 00 explains how it works). The picture is a word made of dots, and every dot asks one small question about a piece of KineticIntro: a dot that gets a wrong answer flies off its letter. Text is drawn with the bitmap font here, so the picture is the same on every computer. Its traps: scenes with every option changed, frames before a scene starts and after it ends, a two-frame exit, masks that haven't started or have long finished, gradient angles all the way round, overlay where one side is transparent, the background and vignette at other sizes, layers at other widths, and whole frames compared with your own pieces drawn by hand. The three questions are below.



In [ ]:
// Checkpoint 09 · KineticIntro. Replace each null with your answer.
MS.answers("09", {
  // 1. For how many frames is "ANIMATE." (a = 40, b = 88) completely opaque, opacity exactly 1?
  q1: null,
  // 2. The light sweep's center s moves from −20% to 130% between frames 140 and 172.
  //    What's the first whole frame at which it has reached the middle (s ≥ 50)?
  q2: null,
  // 3. At full size (1080 wide), how many pixels wide is the margin around the
  //    vignette's hole?
  q3: null,
});

In [ ]:
await checkpoint("09")


### In the real world

- Remotion's [`interpolate`](https://www.remotion.dev/docs/interpolate) with [`Easing.bezier`](https://www.remotion.dev/docs/easing) is exactly your layers 04 and 05, and [`<AbsoluteFill>`](https://www.remotion.dev/docs/absolute-fill) is a full-frame flexbox container; KineticIntro centers its words inside one.
- Chrome draws a Remotion frame with its compositor: each Scene with a transform, filter or opacity can get its own GPU layer, much like your sprites, and blends them in an order very like `renderLayers`.
- CSS's [`isolation: isolate`](https://developer.mozilla.org/en-US/docs/Web/CSS/isolation) makes a stacking context on purpose, so blend modes inside it can't reach outside it.
- Big shadows and blurs are among the most expensive things a page can draw, which is why browsers try to draw them once and reuse the result: "make it once", done for you.

### Going further (optional)

- **Read the original.** The [Compositing and Blending spec](https://www.w3.org/TR/compositing-1/#blending) lists all the separable blend modes with their formulas. Add `screen`, `multiply` and `soft-light` to your `blendOverlay` as options and try each on the sweep.
- Cache the text sprites: they only depend on the word, not on n. How much faster is a frame?
- Render KineticIntro at full size, 1080 × 1920, and time one frame. Which piece takes longest?


## Layer 10 · Export

_A video file is a list of boxes, and you know how to write every byte._

**You'll write:** `u16`, `u32`, `box`, `fullBox`, `stts`, `stsz`, `stco`, `stss`, `mp4` in `src/10-export.js`
**Builds on:** layers 00–09

Your KineticIntro plays in a browser tab. To post it, send it or drop it into an editor, it has to become a file: an MP4. Two very different jobs hide in that word. **Encoding** squeezes each frame from megabytes down to kilobytes; that's a field of its own, and the browser's [WebCodecs](https://www.w3.org/TR/webcodecs/) API does it for you, often on the computer's video hardware. **Packaging** puts the squeezed frames in a file that every player understands, with an index saying where each frame is and when it plays. That part is a few hundred bytes of careful bookkeeping, and you'll write all of it. Then you'll press a button and watch your own file play.

### Under the hood

#### Codec and container

An MP4 is a **container**: a box of boxes that holds the frames, plus a table of contents. The frames inside are compressed by a **codec**, here H.264, the most widely played video format there is. The two are separate: the same H.264 frames could go in a different container, and an MP4 can hold other codecs.

```text
  your frames ──► encoder (H.264, by WebCodecs) ──► chunks ──► container (MP4, by you) ──► file
  RGBA pixels      compresses each frame              one per     where and when each
  (~330 KB each                                       frame,      chunk is, so a player
   at 216 × 384)                                      a few KB    can find and time it
```

#### Boxes all the way down

The format comes from Apple's QuickTime and is standardized as the [ISO base media file format](https://www.iso.org/standard/83102.html) (ISO/IEC 14496-12). It's built from one idea: a **box** (QuickTime calls it an atom) is 4 bytes of size, 4 letters of type, and then its contents, which can be more boxes.

```text
  ┌──────────────┬────────────────┬──────────────────────────────┐
  │ size (u32)   │ type (4 ASCII) │ contents: data, or more boxes│
  │ 00 00 00 18  │ s  t  t  s     │ ...                          │
  └──────────────┴────────────────┴──────────────────────────────┘
     the size counts the whole box, these 8 header bytes included
```

A program that doesn't know a box type can still skip it: read the size, jump ahead. It's one reason the format has lasted since QuickTime's first release in 1991. Many boxes are **full boxes**, whose contents start with a version byte and 3 bytes of flags, so the format can change without breaking old readers.

Every number is **big-endian**: the most significant byte comes first, the way you write digits. 300 as a u32 is `00 00 01 2C`. Your computer's processor is almost certainly little-endian and would store it backwards, which is why `u32` builds the bytes by hand with shifts and masks instead of copying memory. Network protocols use big-endian too; it's often called "network byte order".

```text
  ftyp                      which specs the file follows
  mdat                      the media data: every chunk's bytes, back to back
  moov                      the movie: the table of contents
  └ trak                    one track (we have one: video)
    └ mdia
      └ minf
        └ stbl              the sample table: where and when every frame is
          ├ stsd            how to decode them: "avc1", with the encoder's setup (avcC)
          ├ stts            how long each one lasts
          ├ stsc            which chunk each sample is in
          ├ stsz            how many bytes each one is
          ├ stco            where each chunk starts in the file
          └ stss            which ones are key frames
```

`lib/mp4.js` builds the boilerplate boxes (headers full of zeros and fixed numbers) using your `box`, `fullBox`, `u16` and `u32`. Read it: every field is named. The sample table is yours.

> **Study buddy:** "Ask me questions until I can explain what a box's size counts, why the numbers are big-endian, and how a program can skip a box it doesn't understand. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2010%20%28Export%29%3A%20frames%20to%20MP4%2C%20then%20map%20it%20to%20Remotion.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20what%20a%20box%27s%20size%20counts%2C%20why%20the%20numbers%20are%20big-endian%2C%20and%20how%20a%20program%20can%20skip%20a%20box%20it%20doesn%27t%20understand.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2010%20%28Export%29%3A%20frames%20to%20MP4%2C%20then%20map%20it%20to%20Remotion.%20Here%27s%20what%20I%20want%20help%20with%3A%20Ask%20me%20questions%20until%20I%20can%20explain%20what%20a%20box%27s%20size%20counts%2C%20why%20the%20numbers%20are%20big-endian%2C%20and%20how%20a%20program%20can%20skip%20a%20box%20it%20doesn%27t%20understand.%20No%20code.)

#### Where: sizes and offsets

To show frame 90, a player needs its bytes. `stsz` gives every frame's size and `stco` gives where each **chunk** starts in the file (a chunk is a run of samples stored together; here every frame is its own chunk, which keeps `stsc` a one-liner). The offsets count from the very first byte of the file, so they have to skip the `ftyp`, then the `mdat`'s own 8-byte header, then every chunk before it.

```text
  byte 0                 ftyp.length   + 8
  ┌──────────┬───────────┬─────────────┬────────┬────────┬─ ─ ─┬──────────┐
  │ ftyp     │ mdat hdr  │ chunk 1     │ chunk 2│ chunk 3│     │ moov     │
  └──────────┴───────────┴─────────────┴────────┴────────┴─ ─ ─┴──────────┘
                         ▲ offset 1    ▲ offset 2 = offset 1 + size 1
```

#### When: timescales and runs

MP4 counts time in whole ticks of a **timescale**: ticks per second. This file's media uses 90000, a habit from MPEG broadcasting, because it divides evenly by 24, 25, 30, 50 and 60, so common frame rates get whole-number frame durations. At 30 fps, every frame lasts 3000 ticks. `stts` stores durations as runs, "180 samples of 3000", so a constant frame rate costs 8 bytes no matter how long the video is. That's [run-length encoding](https://en.wikipedia.org/wiki/Run-length_encoding), the same trick as "A × 3" instead of "AAA".

The movie header (`mvhd`) and track header (`tkhd`) keep their own, coarser clock in milliseconds, for showing the length to people.

#### Which: key frames

H.264 saves most of its space by storing most frames as differences from earlier ones. A **key frame** stands alone; the frames after it are built from it. A player can only start decoding at a key frame, so jumping to 0:04 means finding the key frame at or before it. `stss` lists them, by sample number, counting from 1. The provided encoder makes one every two seconds: fewer key frames make a smaller file, and more make seeking quicker.

#### The order of the boxes

Your file puts `mdat` before `moov`. That's the easy order: when you write the offsets you already know exactly where everything is. It's also how a camera records: it can't write the table of contents until it stops filming. Its one downside is that a player streaming the file over a network has to fetch the end before it can start. Moving `moov` to the front ("fast start") fixes that, but then the offsets have to skip the `moov` too.

### Your turn

Work through `src/10-export.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **u16 and u32.** How do you get the top byte of a 32-bit number? Which operator moves bits right without bringing the sign along?
2. **box.** What's the size of a box with no contents? How do you know the total size before you copy the parts in?
3. **fullBox.** Can you build it out of `box`?
4. **stts.** Walking through the durations, when do you start a new run?
5. **stsz, stco, stss.** In what order do the count and the list go? What number does the first sample have?
6. **mp4.** Which boxes can you build before you know the offsets? Where does the first chunk start? Which numbers are in 90000ths of a second and which in milliseconds?

Run the tests after each function. A failing test tells you exactly what it expected. `MS.parseMP4(file)` (provided) reads your file back as a tree of boxes, and `MS.findBox(tree, "moov/trak/mdia/minf/stbl")` finds one: use them to look at what you made.

Once it works, the test pictures show every byte of a small MP4 as a pixel, colored by its box, and your KineticIntro with an **Export MP4** button. In a browser, press it: the frames are rendered by your code, encoded by the browser, packaged by your `mp4()`, and played back on the page. That's your video file.

> **Study buddy:** "Hand me the first 64 bytes of an MP4 as hex and let me find the box boundaries, sizes and types myself. Check my reading, but don't read it for me. No code." [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2010%20%28Export%29%3A%20frames%20to%20MP4%2C%20then%20map%20it%20to%20Remotion.%20Here%27s%20what%20I%20want%20help%20with%3A%20Hand%20me%20the%20first%2064%20bytes%20of%20an%20MP4%20as%20hex%20and%20let%20me%20find%20the%20box%20boundaries%2C%20sizes%20and%20types%20myself.%20Check%20my%20reading%2C%20but%20don%27t%20read%20it%20for%20me.%20No%20code.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2010%20%28Export%29%3A%20frames%20to%20MP4%2C%20then%20map%20it%20to%20Remotion.%20Here%27s%20what%20I%20want%20help%20with%3A%20Hand%20me%20the%20first%2064%20bytes%20of%20an%20MP4%20as%20hex%20and%20let%20me%20find%20the%20box%20boundaries%2C%20sizes%20and%20types%20myself.%20Check%20my%20reading%2C%20but%20don%27t%20read%20it%20for%20me.%20No%20code.)



### ✏️ Your code for layer 10

This cell is your `src/10-export.js`. Write the functions, run the cell, then run the check below it.

**Stuck?** Ask a study buddy that gives hints, not answers: [Ask Claude](https://claude.ai/new?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2010%20%28Export%29%3A%20frames%20to%20MP4%2C%20then%20map%20it%20to%20Remotion.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.) · [Ask ChatGPT](https://chatgpt.com/?q=You%27re%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20%28github.com%2Fnalediym%2Fmotion-from-scratch%29%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Never%20give%2C%20confirm%20or%20rule%20out%20checkpoint%20answers.%20Ask%20me%20questions%20first.%20If%20I%27m%20still%20stuck%2C%20climb%20this%20hint%20ladder%20one%20rung%20at%20a%20time%2C%20never%20past%20the%20top%3A%201%29%20ask%20what%20I%20expected%2C%20what%20happened%20and%20what%20I%27ve%20tried%3B%202%29%20point%20at%20a%20failing%20test%27s%20expected%20vs%20actual%20values%3B%203%29%20ask%20a%20concept%20question%20and%20point%20to%20the%20chapter%20section%3B%204%29%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%2C%20using%20different%20numbers%3B%205%29%20point%20at%20the%20line%20to%20look%20at%20and%20ask%20about%20it.%20Explain%20what%27s%20happening%20under%20the%20hood%20%28bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it%29.%20I%27m%20on%20layer%2010%20%28Export%29%3A%20frames%20to%20MP4%2C%20then%20map%20it%20to%20Remotion.%20Start%20by%20asking%20what%20I%27m%20working%20on.%20I%27ll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
// Layer 10 · Export — frames to MP4, then map it to Remotion.
// Read the chapter first (docs/10-export.md). Each comment below is a spec; the tests check it.
(function (MS) {
  // An MP4 file is a list of boxes, and a box can hold other boxes. All numbers
  // are big-endian: the most significant byte comes first.
  // lib/mp4.js (provided) builds the boilerplate boxes with your functions;
  // read it to see each field.

  /** n (a whole number from 0 to 65535) as 2 bytes, biggest first, in a Uint8Array. */
  function u16(n) {
    throw MS.todo("u16");
  }

  /** n (a whole number from 0 to 2³² − 1) as 4 bytes, biggest first, in a Uint8Array. */
  function u32(n) {
    throw MS.todo("u32");
  }

  /**
   * A box, as a Uint8Array: its size in bytes as a u32 (the whole box,
   * these 4 bytes included), its type as 4 ASCII characters, then each part
   * in order. Parts are Uint8Arrays or plain arrays of byte values.
   *   box("abcd", [1, 2]) → [0, 0, 0, 10, 97, 98, 99, 100, 1, 2]
   */
  function box(type, ...parts) {
    throw MS.todo("box");
  }

  /**
   * A "full box": a box whose contents start with a version byte and 3 bytes
   * of flags (biggest first), then the parts.
   */
  function fullBox(type, version, flags, ...parts) {
    throw MS.todo("fullBox");
  }

  /**
   * The time-to-sample box: how long each sample (frame) lasts, in the
   * track's timescale. A full box "stts", version 0, flags 0, holding the
   * number of entries, then for each run of samples in a row with the same
   * duration: how many (u32) and the duration (u32).
   *   stts([3000, 3000, 1500]) holds 2 entries: (2, 3000) and (1, 1500)
   */
  function stts(durations) {
    throw MS.todo("stts");
  }

  /**
   * The sample size box: a full box "stsz", version 0, flags 0, holding a
   * u32 0 (meaning "the sizes differ, here's a list"), the number of samples,
   * then each sample's size in bytes, all u32.
   */
  function stsz(sizes) {
    throw MS.todo("stsz");
  }

  /**
   * The chunk offset box: a full box "stco", version 0, flags 0, holding the
   * number of chunks, then where each chunk's first byte is in the file
   * (counting from the file's first byte as 0), all u32.
   */
  function stco(offsets) {
    throw MS.todo("stco");
  }

  /**
   * The sync sample box: which samples are key frames, where a player can
   * start decoding. `keys` has one true or false per sample. A full box
   * "stss", version 0, flags 0, holding how many keys there are, then each
   * key sample's number, counting the first sample as 1, all u32.
   */
  function stss(keys) {
    throw MS.todo("stss");
  }

  /**
   * A whole MP4 file, as a Uint8Array, for chunks from the encoder:
   * [{ data (Uint8Array), key (boolean), timestamp, duration }], with
   * timestamp and duration in microseconds, one chunk per frame, in order.
   * opts: { width, height, description } (description: the encoder's avcC bytes).
   * The file is three boxes in a row:
   *   ftyp   MS.MP4.ftyp()
   *   mdat   every chunk's data, back to back, in order
   *   moov   mvhd, then trak:
   *            tkhd, then mdia:
   *              mdhd, hdlr, then minf:
   *                vmhd, dinf, then stbl: stsd, stts, stsc, stsz, stco, stss
   * Every MS.MP4 builder is in lib/mp4.js; the rest are your functions.
   * Time: the media (mdhd, stts) counts in 90000ths of a second; a chunk's
   * duration there is Math.round(duration × 90000 ÷ 1000000), and mdhd's
   * duration is their total. The movie (mvhd) counts in milliseconds
   * (timescale 1000): its duration, and tkhd's, is Math.round(total × 1000 ÷ 90000).
   * Each chunk is its own chunk in stco: its offset is where its data starts
   * inside the file.
   */
  function mp4(chunks, opts) {
    throw MS.todo("mp4");
  }

  Object.assign(MS, { u16, u32, box, fullBox, stts, stsz, stco, stss, mp4 });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
await check("10")

### Predict, then run

Guess first, then run.



In [ ]:
{ // the braces let you re-run this cell
// Predict 1: what are the 4 bytes of u32(1000)?
const myGuess = null; // TODO: your guess, as [a, b, c, d]
MS.reveal("u32(1000)", myGuess, Array.from(MS.u32(1000)));
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 2: 180 frames that all last 3000 ticks. How many bytes is the whole stts box?
const myGuess = null; // TODO: your guess
MS.reveal("stts bytes", myGuess, MS.stts(new Array(180).fill(3000)).length);
}

In [ ]:
{ // the braces let you re-run this cell
// Predict 3: three chunks of 1000, 200 and 200 bytes. Where in the file does the second one start?
const myGuess = null; // TODO: your guess, a byte number
const chunks = [1000, 200, 200].map((n, i) => ({ data: new Uint8Array(n), key: i === 0, timestamp: i * 33333, duration: 33333 }));
const file = MS.mp4(chunks, { width: 16, height: 16, description: new Uint8Array([1, 66, 0, 30]) });
const stco = MS.findBox(MS.parseMP4(file), "moov/trak/mdia/minf/stbl/stco");
MS.reveal("the second chunk's offset", myGuess, new DataView(stco.body.buffer, stco.body.byteOffset).getUint32(12));
}


<details><summary>Answers</summary>

1. [0, 0, 3, 232]. 1000 is 3 × 256 + 232. The two top bytes are 0, and they still have to be there: a u32 is always 4 bytes.
2. 24. 8 for the box header, 4 for version and flags, 4 for the entry count, and one entry of 8 bytes: (180, 3000). An hour of 30 fps video would make it exactly the same size.
3. 1040. The ftyp is 32 bytes (8 of header, then "isom", a version and four 4-letter brands), the mdat's header is 8 more, and the first chunk is 1000.

</details>

### Break it

Once your tests pass: make each TODO change, run the experiment before and after, explain the difference, then undo it.

#### 1. Little-endian



In [ ]:
{ // the braces let you re-run this cell
// Break it 1: what happens if the bytes are backwards?
// TODO(break-it), in src/10-export.js → u32:
//   write the smallest byte first.
const chunks = [400, 120].map((n, i) => ({ data: new Uint8Array(n), key: i === 0, timestamp: 0, duration: 33333 }));
const file = MS.mp4(chunks, { width: 16, height: 16, description: new Uint8Array([1, 66, 0, 30]) });
console.log("first 8 bytes: " + Array.from(file.slice(0, 8)).join(" "));
try { console.log("boxes: " + MS.parseMP4(file).map(b => b.type).join(", ")); }
catch (e) { console.log("can't read it: " + e.message); }
// TODO: what size does the first box claim to be after the change? What would a video player do with that?
}


#### 2. Forget the mdat's header



In [ ]:
{ // the braces let you re-run this cell
// Break it 2: what is an offset measured from?
// TODO(break-it), in src/10-export.js → mp4:
//   start the first chunk's offset at the end of the ftyp, without the mdat's 8 header bytes.
const chunks = [[9, 9, 9, 9], [7, 7, 7]].map((b, i) => ({ data: new Uint8Array(b), key: i === 0, timestamp: 0, duration: 33333 }));
const file = MS.mp4(chunks, { width: 16, height: 16, description: new Uint8Array([1, 66, 0, 30]) });
const stco = MS.findBox(MS.parseMP4(file), "moov/trak/mdia/minf/stbl/stco");
const first = new DataView(stco.body.buffer, stco.body.byteOffset).getUint32(8);
console.log("chunk 1 is at byte " + first + "; the 4 bytes there are " + Array.from(file.slice(first, first + 4)).join(" "));
// TODO: before the change they're 9 9 9 9. After it, what is the player reading instead of the frame?
}


#### 3. Count key frames from 0



In [ ]:
{ // the braces let you re-run this cell
// Break it 3: does it matter where counting starts?
// TODO(break-it), in src/10-export.js → stss:
//   number the samples from 0 instead of 1.
const b = MS.stss([true, false, false, true]);
const v = new DataView(b.buffer, b.byteOffset);
console.log("stss lists: " + [1, 2].map(i => v.getUint32(12 + 4 * i)).join(", "));
// TODO: which frames does a player now think are key frames? What happens when someone seeks to 0:02 and it starts decoding at a frame that isn't one?
}


### Checkpoint

When all 15 tests pass, the checkpoint unlocks (layer 00 explains how it works). The picture is a word made of dots, and every dot asks one small question about the bytes of an MP4: a dot that gets a wrong answer flies off its letter. Its traps: numbers whose bytes are all different, numbers past 2¹⁶ and up to 2³² − 1, boxes inside boxes inside boxes, a part that's empty, durations that alternate or run out at the end, empty tables, a single chunk, frames at 24 fps whose durations round, and key frames that aren't the first. The three questions are below.



In [ ]:
// Checkpoint 10 · Export. Replace each null with your answer.
MS.answers("10", {
  // 1. How many bytes long is the mvhd box that MS.MP4.mvhd() makes?
  q1: null,
  // 2. NTSC video runs at 29.97 fps: each frame lasts 1001 ÷ 30000 of a second.
  //    How many ticks is that at timescale 90000?
  q2: null,
  // 3. KineticIntro has 180 frames, with a key frame every 60 frames starting at the
  //    first. Which numbers does stss list? Answer as an array.
  q3: null,
});

In [ ]:
await checkpoint("10")


### In the real world

You've built, from single pixels up, every piece of a motion-graphics engine. Here's where each one lives in Remotion:

| You wrote | Layer | In Remotion |
|---|---|---|
| `createComposition`, `renderFrame` | 03 | [`<Composition>`](https://www.remotion.dev/docs/composition), and each frame a headless Chrome renders |
| `frameAt`, `MS.play` | 03 | [`<Player>`](https://www.remotion.dev/docs/player), [`useCurrentFrame()`](https://www.remotion.dev/docs/use-current-frame) |
| `interpolate`, `interpolateColors` | 04 | [`interpolate()`](https://www.remotion.dev/docs/interpolate), [`interpolateColors()`](https://www.remotion.dev/docs/interpolate-colors) |
| `cubicBezier` | 05 | [`Easing.bezier()`](https://www.remotion.dev/docs/easing) |
| `spring` | 06 | [`spring()`](https://www.remotion.dev/docs/spring) |
| matrices, `drawTransformed`, `blur` | 07 | CSS `transform` and `filter: blur()`, drawn by Chrome |
| `sequence`, `series`, `random`, `textSprite` | 08 | [`<Sequence>`](https://www.remotion.dev/docs/sequence), [`<Series>`](https://www.remotion.dev/docs/series), [`random()`](https://www.remotion.dev/docs/random), web fonts |
| `kineticIntro` | 09 | the KineticIntro component itself |
| `mp4` | 10 | [`renderMedia()`](https://www.remotion.dev/docs/renderer/render-media), which hands the frames to FFmpeg to encode and package |

- FFmpeg's `-movflags +faststart` moves the `moov` box to the front after writing, the "fast start" from the chapter.
- Files over 4 GB need offsets bigger than a u32 can hold: the `co64` box is `stco` with u64 offsets.
- Streaming video (YouTube, Netflix) uses **fragmented MP4**: many small `moof` + `mdat` pairs, each with its own little sample table, so a player can start before the file ends.

### Going further (optional)

- **Read the original.** ISO/IEC 14496-12 is long, but its box definitions are short and regular. Find `stts` and `stss` in it and compare them with your code. Apple's [QuickTime File Format](https://developer.apple.com/documentation/quicktime-file-format) documentation describes the same boxes as "atoms", with diagrams.
- Make a fast-start file: build the `moov` first with placeholder offsets, measure it, then fix the offsets. Check it still plays.
- Open your exported file in a hex editor (or `xxd` in a terminal) and find the `moov`, the `stss` and your first key frame by hand.
